# || NEMO Workstation ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

please cite: K.Andreadis _et al._ "NEMO: Mesh-Based Tangential Nematic Field, Defect, and Morphology Analysis in Volumetric Microscopy Data" (2026, _in preparation_)

In [ ]:
# Import custom scripts
from importlib import reload

from scripts import analysis, datahandler, visuals, simulation

for module in (analysis, datahandler, visuals, simulation):
    reload(module)

# Import python essentials
import os
import numpy as np
import matplotlib.pyplot as plt
import trimesh

In [ ]:
# Initialise Napari viewer once
visuals.view_mesh([])

# --Import Image--

In [ ]:
# ==== Choose Image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/nemo_debug.tif'

print(f"Selected image path: {img_path}")

# ==== Choose Time Step and Channel ====
t_select = 0
c_select = 0

# ==== [Optional] Reduce Resolution ====
z_reduce_factor = 1  # 1 means no reduction
xy_reduce_factor = 1  # 1 means no reduction

# ==== [Optional] Normalise Intensities to [0, 1] ====
normalise_intensities = False  # can be set to False

# ==== [Optional] Overwrite Scaling with FIJI Values ====
custom_scaling = None  # please use (z, y, x)

# ==== [Optional] Overwrite unit with FIJI Values ====
custom_unit = "um"  # as string

# ==== Load Image ====
img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_select, c_sel_idx=c_select, reduce_xy=xy_reduce_factor,
                                     reduce_z=z_reduce_factor, norm_vals=normalise_intensities,
                                     custom_scaling=custom_scaling, custom_unit=custom_unit)
if img_load is not None:
    img_raw, img_dim, img_scale, img_unit = img_load

    # ==== Create Folder Structure ====
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
    # resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")

    # ==== Plot Image Slices and Max Projections ====
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, max_proj=True, cmap='Greens',
                     savefig=os.path.join(resfig_dir, "sliced_maxproj_raw.png"))
    # z_i, y_i, x_i = int(200 / img_scale[0]), int(570 / img_scale[1]), int(455 / img_scale[2])
    z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, x_i=x_i, y_i=y_i, z_i=z_i,
                     savefig=os.path.join(resfig_dir, "sliced_raw.png"), cmap="Greens_r")
else:
    # ==== Default scenario ====
    img_raw, img_dim, img_scale, img_unit = np.zeros((1, 1, 1)), (1, 1, 1), (1, 1, 1), "?"
    resdata_dir, resfig_dir = None, None

In [ ]:
# ==== 3D Render Image ====
visuals.view_img(img_list=[img_raw], title_list=["Raw Image"], scale=img_scale)

## Channel Composite Viewer

In [ ]:
# ==== Choose Time Point ====
t_select = 0
dims = analysis.load_img_dimensions(img_path)
num_channels = dims["C"]
channel_colors = ["Greens", "Reds", "Blues"]
channel_labels = ["Channel 1", "Channel 2", "Channel 3"]
composite_stack = []

for c_i in range(num_channels):
    # ==== Choose Time Step and Channel ====
    c_select = c_i

    # ==== Load Image ====
    img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_select,
                                         c_sel_idx=c_select, reduce_xy=1, reduce_z=1, norm_vals=False,
                                         custom_scaling=None, custom_unit="um")
    img_raw_i, img_dim_i, img_scale_i, img_unit_i = img_load
    composite_stack.append(img_raw_i)

    # # ==== Plot Image Slices and Max Projections ====
    visuals.plot_img(img=img_raw_i, scale=img_scale_i, unit=img_unit_i, max_proj=True, cmap=f"{channel_colors[c_i]}",
                     savefig=os.path.join(resfig_dir, f"c={c_i}_sliced_maxproj_raw.png"))
    z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
    visuals.plot_img(img=img_raw_i, scale=img_scale_i, unit=img_unit_i, x_i=x_i, y_i=y_i, z_i=z_i,
                     cmap=f"{channel_colors[c_i]}_r", savefig=os.path.join(resfig_dir, f"c={c_i}_sliced_raw.png"))
# ==== 3D Render Image ====
visuals.view_img(img_list=composite_stack, title_list=channel_labels, color_list=[f"{i}_r" for i in channel_colors],
                 scale=img_scale, opacity_list=[0.8 for i in channel_colors])

## Live Channel Viewer

In [ ]:
# ==== Choose Channel ====
c_select = 0
dims = analysis.load_img_dimensions(img_path)
num_timepoints = dims["T"]
time_stack = []

for t_i in range(num_timepoints):
    # ==== Load Image ====
    img_load = analysis.load_img_virtual(path=img_path, t_sel_idx=t_i,
                                         c_sel_idx=c_select, reduce_xy=1, reduce_z=1, norm_vals=False,
                                         custom_scaling=None, custom_unit="um")
    img_raw_i, img_dim_i, img_scale_i, img_unit_i = img_load
    time_stack.append(img_raw_i)
time_stack = np.stack(time_stack, axis=0)
# ==== 3D Render Image ====
visuals.view_img(img_list=[time_stack], scale=img_scale)

# -- Create Mesh --

## |1| Image Blur & Threshold

In [ ]:
# ==== Blur Image ====
sigma = 6
sigma_ = analysis.rescale_val_xyz(val=sigma, scale=img_scale)
img_blur = analysis.gaussian_blur(img=img_raw, sigma=sigma_, renorm=False)

# ==== Plot Image Slices ====
visuals.plot_img(img=img_blur, scale=img_scale, unit=img_unit, cmap="inferno",
                 savefig=os.path.join(resfig_dir, "sliced_blur.png"))

# ==== 3D Render Image ====
# visuals.view_img([img_blur], scale=img_scale, title_list=["Blurred Image"])

In [ ]:
# ==== Yen Threshold Image ====
# img_thresh_val = np.min([analysis.yen_thresh(img_blur[:, :, img_dim[2] // i]) for i in np.arange(2, 6)])
img_thresh_val = np.min(
    [analysis.yen_thresh(img_blur[:, :, img_dim[2] // 2]),
     analysis.yen_thresh(img_blur[:, img_dim[1] // 2, :]),
     analysis.yen_thresh(img_blur[img_dim[0] // 2, :, :])]) * 0.1
# img_thresh_val = 7

# ==== Binarise Image using Threshold ====
img_thresh = analysis.thresh_img(img_blur, img_thresh_val)

# ==== Plot Image Slices ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, savefig=os.path.join(resfig_dir, "sliced_thresh.png"),
                 cmap="inferno", thresh_mask=img_thresh)

In [ ]:
# ==== 3D Render Image ====
visuals.view_img([img_raw, img_thresh], scale=img_scale, title_list=["Raw Image", "Thresholded Image"],
                 color_list=["Greens_r", "Blues_r"], opacity_list=[1.0, 0.7])

In [ ]:
# # ==== [Optional] Fill Holes in Binary Image ====
# img_thresh = analysis.fill_holes_img(img_thresh)
#
# # ==== Plot Image Slices ====
# visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
#                  savefig=os.path.join(resfig_dir, "sliced_thresh.png"),
#                  cmap="inferno", thresh_mask=img_thresh)

In [ ]:
# # ==== [Optional] Save Thresholded Raw Image as .tiff ====
# visuals.view_img([img_raw, img_thresh], opacity_list=[1.0, 0.6], color_list=["green", "Blues"], scale=img_scale,
#                  title_list=["Raw Image", "Thresholded Image"])
# img_thresh = analysis.thresh_img(img_raw, img_thresh_val, keep_values_above=True)
# datahandler.save_tiff(img_thresh, filepath=os.path.join(resfig_dir, "thresh_img.tiff"))

## |2| Surface Mesh Extraction

In [ ]:
# ==== Segment Surface Mesh(es) ====
mcub_res = 2
full_mesh = analysis.marching_cubes(img=img_thresh, scale=img_scale, level=0.5, step_size=mcub_res)

# ==== Select INNER / OUTER Mesh ====
mesh_sel_mask = np.einsum('ij,ij->i', full_mesh.vertices - np.mean(full_mesh.vertices, axis=0),
                          full_mesh.vertex_normals) > 0

# ==== Select TOP / BOTTOM Mesh ====
# mesh_sel_mask = np.einsum('ij,ij->i', np.array([[1, 0, 0] for _ in range(len(full_mesh.vertices))]),
#                           full_mesh.vertex_normals) < 0

# ==== Apply Sub-Mesh Selection ====
inner_mesh = analysis.sel_submesh(mesh=full_mesh, mask=mesh_sel_mask)
outer_mesh = analysis.sel_submesh(mesh=full_mesh, mask=~mesh_sel_mask)
# inner_mesh = analysis.find_connected_meshes(mesh=full_mesh)[0]

# ==== Save Mesh(es) ====
datahandler.save_mesh(full_mesh, os.path.join(resdata_dir, "full_mesh.ply"))
datahandler.save_mesh(inner_mesh, os.path.join(resdata_dir, "inner_mesh.ply"))
datahandler.save_mesh(outer_mesh, os.path.join(resdata_dir, "outer_mesh.ply"))
print(
    f"Number of vertices: #INNER = {inner_mesh.vertices.shape[0]} + #OUTER = {outer_mesh.vertices.shape[0]} == #FULL = {full_mesh.vertices.shape[0]}!")

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_raw_full-mesh.png"), meshes=[inner_mesh, outer_mesh],
                 mesh_colors=["red", "blue"])

In [ ]:
# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[outer_mesh, inner_mesh], mesh_colors=["white", "red"],
#                   mesh_titles=["Outer Mesh", "Inner Mesh"], mesh_opacities=[0.3, 0.3],
#                   img=img_raw, img_opacity=0.5, scale=img_scale)
visuals.view_mesh(mesh_list=[full_mesh], mesh_colors=["white"],
                  mesh_titles=["Full Mesh"], mesh_opacities=[1.0], vec_freq=5,
                  img=img_raw, img_opacity=0.9, scale=img_scale)

## |3| Mesh Processing

In [ ]:
# ==== Smooth Mesh(es) ====
smooth_factor = 0.001
smooth_iterations = 100
full_mesh_smooth = analysis.taubin_smooth_mesh(mesh=full_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)
inner_mesh_smooth = analysis.taubin_smooth_mesh(mesh=inner_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)
outer_mesh_smooth = analysis.taubin_smooth_mesh(mesh=outer_mesh, n_iter=smooth_iterations, pass_band=smooth_factor)

# ==== Sub-Sample Mesh(es) ====
# full_mesh_smooth = analysis.subdivide_mesh(mesh=full_mesh_smooth, max_edge=2)
# inner_mesh_smooth = analysis.subdivide_mesh(mesh=inner_mesh_smooth, max_edge=2)
# outer_mesh_smooth = analysis.subdivide_mesh(mesh=outer_mesh_smooth, max_edge=2)

# ==== Save Mesh(es) ====
datahandler.save_mesh(full_mesh_smooth, os.path.join(resdata_dir, "full_mesh_smooth.ply"))
datahandler.save_mesh(inner_mesh_smooth, os.path.join(resdata_dir, "inner_mesh_smooth.ply"))
datahandler.save_mesh(outer_mesh_smooth, os.path.join(resdata_dir, "outer_mesh_smooth.ply"))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_full-mesh.png"), meshes=[full_mesh, full_mesh_smooth],
                 mesh_colors=["black", "purple"])
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_inner-mesh.png"),
                 meshes=[inner_mesh, inner_mesh_smooth], mesh_colors=["black", "red"])
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_outer-mesh.png"),
                 meshes=[outer_mesh, outer_mesh_smooth], mesh_colors=["black", "blue"])

In [ ]:
# ==== 3D Render Mesh(es) ====
visuals.view_mesh(mesh_list=[outer_mesh_smooth, inner_mesh_smooth], mesh_colors=["white", "red"],
                  mesh_titles=["Smooth Outer Mesh", "Smooth Inner Mesh"], mesh_opacities=[0.3, 0.3],
                  img=img_raw, img_opacity=0.5, scale=img_scale)

# visuals.view_mesh(mesh_list=[inner_mesh, inner_mesh_smooth], mesh_colors=["grey", "red"],
#                   mesh_titles=["Inner Mesh", "Smooth Inner Mesh"], mesh_opacities=[0.3, 0.3])
# visuals.view_mesh(mesh_list=[full_mesh, full_mesh_smooth], mesh_colors=["grey", "red"],
#                   mesh_titles=["Full Mesh", "Smooth Full Mesh"], mesh_opacities=[0.3, 0.3], img=img_raw,
#                   scale=img_scale)

### |3.1| Select Largest Mesh

In [ ]:
full_mesh_smooth_subset_all = analysis.find_connected_meshes(mesh=full_mesh_smooth)
sizes = [mesh.vertices.shape[0] for mesh in full_mesh_smooth_subset_all]
full_mesh_smooth_subset = full_mesh_smooth_subset_all[np.argmax(sizes)]
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_smooth_full-mesh_sub.png"), meshes=[full_mesh_smooth_subset],
                 mesh_colors=["purple"])

In [ ]:
visuals.view_mesh(mesh_list=[full_mesh_smooth, full_mesh_smooth_subset], mesh_colors=["white", "red"],
                  mesh_titles=["Smooth Full Mesh", "Largest Subset of Full Mesh"], mesh_opacities=[0.8, 1.0],
                  vec_freq=5,
                  img=img_raw, img_opacity=0.5, scale=img_scale)

### |3.2| EMBL Sphere Fit

In [ ]:
mesh_to_fit = full_mesh_smooth.copy()
# ==== Fit Sphere ====
sphere_params = analysis.fit_sphere(points=mesh_to_fit.vertices)
sphere_x0, sphere_y0, sphere_z0, sphere_radius = sphere_params
sphere_mesh = trimesh.creation.icosphere(radius=sphere_radius, subdivisions=8)
sphere_mesh.vertices += [sphere_x0, sphere_y0, sphere_z0]
datahandler.save_array(np.array(sphere_params)[:, np.newaxis].T, "sphere_fit", header="x,y,z,radius",
                       folderpath=resdata_dir)
# ==== Crop (Part of) Sphere ====
seg_fit_crop_cap_angle = 120.0
print(f">> Cropping sphere to {seg_fit_crop_cap_angle} degrees cap...")
sphere_crop_mask = ((sphere_mesh.vertices[:, 0] - sphere_x0) / np.linalg.norm(
    sphere_mesh.vertices - [sphere_x0, sphere_y0, sphere_z0], axis=1)) >= np.cos(
    np.radians(seg_fit_crop_cap_angle))
sphere_mesh_cropped = analysis.sel_submesh(mesh=sphere_mesh, mask=sphere_crop_mask)
print(f"Num of sphere vertices: {sphere_mesh_cropped.vertices.shape[0]}")

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sphere-fit.png"), cmap="Greens_r", show_mesh_normals=True,
                 meshes=[sphere_mesh_cropped], mesh_alpha=1.0)

In [ ]:
# ==== 3D Render Mesh(es) ====
visuals.view_mesh([sphere_mesh, sphere_mesh_cropped], mesh_opacities=[0.5, 1.0], mesh_colors=["white", "red"],
                  img=img_raw, scale=img_scale)

## |5| Save Sampling Mesh

In [ ]:
# ==== Select Sampling Mesh ====
sampl_mesh = full_mesh_smooth_subset.copy()  # full_mesh_smooth, outer_mesh_smooth, inner_mesh_smooth

# ==== [Optional] Merge Meshes Instead ====
# mesh_merge_1 = datahandler.load_mesh(os.path.join(resdata_dir, "inner_mesh_smooth.ply"), recalc_normals=False)
# mesh_merge_2 = datahandler.load_mesh(os.path.join(resdata_dir, "outer_mesh_smooth.ply"), recalc_normals=False)
# mesh_merge_1.vertex_normals = -1 * mesh_merge_2.vertex_normals
# sampl_mesh = trimesh.util.concatenate([mesh_merge_1, mesh_merge_2])

# ==== Save Sampling Mesh ====
datahandler.save_mesh(sampl_mesh, os.path.join(resdata_dir, "sampling_mesh.ply"))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=1, show_mesh_normals=True,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sampling-mesh.png"))
mesh_slice_max = np.array([img_scale[i] * img_dim[i] for i in range(len(img_scale))]).max() / 2
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=mesh_slice_max, mesh_alpha=0.05,
                 savefig=os.path.join(resfig_dir, "sliced"
                                                  "_raw_sampling-mesh_maxproj.png"))
print(f"Number of sampling points: {len(sampl_mesh.vertices)} !")

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[0.4],
#                   img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=1)

# -- Load Mesh --

In [ ]:
# ==== Load Sampling Mesh ====
mesh_path = os.path.join(resdata_dir, "sampling_mesh.ply")
sampl_mesh = datahandler.load_mesh(mesh_path, recalc_normals=True, clean=False)
print(f"Number of sampling vertices: {len(sampl_mesh.vertices)} !")
sampl_mesh.vertex_normals = -1 * sampl_mesh.vertex_normals

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=1, show_mesh_normals=True,
                 savefig=os.path.join(resfig_dir, "sliced_raw_sampling-mesh.png"))
mesh_slice_max = np.array([img_scale[i] * img_dim[i] for i in range(len(img_scale))]).max() / 2
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[sampl_mesh],
                 slice_depth=mesh_slice_max, mesh_alpha=0.05,
                 savefig=os.path.join(resfig_dir, "sliced"
                                                  "_raw_sampling-mesh_maxproj.png"))
print(f"Number of sampling points: {len(sampl_mesh.vertices)} !")

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[0.4],
#                   img=img_raw, scale=img_scale, vec_freq=400, hide_vectors=True, vec_length=20)
# visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[1.0],
#                   img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=5, vec_edge_width=0.1)

In [ ]:
visuals.view_mesh(mesh_list=[sampl_mesh], mesh_titles=["Mesh"], mesh_colors=["white"], mesh_opacities=[1.0],
                  img=img_raw, scale=img_scale, vec_freq=100, hide_vectors=False, vec_length=10, vec_edge_width=0.2)

# -- Mesh Analysis --

## |1| Dual Mesh Distance/Thickness

In [ ]:
# ==== Select Mesh(es) ====
mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, "inner_mesh_smooth.ply"), recalc_normals=True, clean=False)
mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, "outer_mesh_smooth.ply"), recalc_normals=True, clean=False)
mesh_2.vertex_normals = -1 * mesh_2.vertex_normals

datahandler.save_mesh(mesh_1, os.path.join(resdata_dir, "mesh_1_morph.ply"))
datahandler.save_mesh(mesh_2, os.path.join(resdata_dir, "mesh_2_morph.ply"))

# ==== Plot Image Slices with Mesh Overlay ====
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[mesh_1, mesh_2],
                 mesh_colors=["red", "blue"], slice_depth=1, show_mesh_normals=True, normal_scale=0.05)

# ==== 3D Render Mesh(es) ====
# visuals.view_mesh(mesh_list=[mesh_1, mesh_2],
#                   mesh_colors=["Green", "Red"],
#                   mesh_titles=["MESH 1", "MESH 2"], img=img_raw, img_opacity=0.5, scale=img_scale)

In [ ]:
# ==== Calculate Inter-Mesh Distance ====
# thickness_crop_range = [0, 30]
thickness_crop_range = None
thickness_sampl_number = 2000
dist_vals, dist_idxs = analysis.inter_dist_mesh(mesh_1=mesh_1, mesh_2=mesh_2, num_sample=thickness_sampl_number,
                                                crop_range=thickness_crop_range, debug=True, allow_multiple_hits=False)
full_dist_vals = analysis.interpolate_on_mesh(mesh_1, dist_idxs, dist_vals, k=10)
# ==== Save Inter-Mesh Distance ====
datahandler.save_array(full_dist_vals, "thickness", header=f"dist ({img_unit})", folderpath=resdata_dir)

# ==== Plot Inter-Mesh Distance ====
visuals.plot_hist(array=full_dist_vals, title=f"Thickness AVG = {full_dist_vals.mean():.2e} {img_unit}",
                  xlim=thickness_crop_range,
                  savefig=os.path.join(resfig_dir, "thickness_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_dist_vals, cmap="Spectral",
                         hexsize=100, cmap_label=f"Thickness ({img_unit})",
                         savefig=os.path.join(resfig_dir, "thickness.png"), figsize=(12, 5))

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 4), sharey=True)
ax[0].scatter(mesh_1.vertices[:, 0], full_dist_vals, marker="x")
ax[1].scatter(mesh_1.vertices[:, 1], full_dist_vals, marker="x")
ax[2].scatter(mesh_1.vertices[:, 2], full_dist_vals, marker="x")
ax[0].set_xlabel(f"x ({img_unit})")
ax[1].set_xlabel(f"y ({img_unit})")
ax[2].set_xlabel(f"z ({img_unit})")
ax[0].set_ylabel(f"Thickness ({img_unit})")
plt.show()

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1, mesh_2],
                                   [visuals.color_scalar(full_dist_vals, normalise=True, cmap="coolwarm"),
                                    "white"], mesh_blending_list=["opaque", "translucent"],
                                   mesh_opacity_list=[1.0, 0.3])

## |2| Dual Mesh Gaussian & Mean Curvature

In [ ]:
# ==== Define Crop Range of Gauss & Mean Curvature ====
gauss_exp = 1 / (np.ptp(mesh_1.vertices, axis=0).mean() / 2) ** 2
mean_exp = 1 / (np.ptp(mesh_1.vertices, axis=0).mean() / 2)
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2), "
      f"Mean should be around {mean_exp:.2e} (1/{img_unit})")
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
mean_order_min, mean_order_max = round(np.log10(mean_exp)) - 1, round(np.log10(mean_exp)) + 1
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]
mean_crop_range = [0.01 * 10 ** mean_order_min, 10 ** mean_order_max]

# gauss_crop_range, mean_crop_range = None, None

# ==== Define Number of Random Calculation Selection ====
curvature_num_samples = 3000

# ==== Define Number of Nearest Neighbours to use for surface fit ====
curvature_patch_info = ["nearest", 20]
# curvature_patch_info = ["radius", 100]
# ==== [Optional] Exclude Boundary Vertices ====
curvature_filter_boundary = False
curvature_filter_boundary_factor = 0.0  # exclude strength between 0 (max) and 1 (no exclusion)

### |2.1| Curvature Mesh #1

In [ ]:
# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_1, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=gauss_crop_range, mean_crop_range=mean_crop_range)

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_1 = analysis.interpolate_on_mesh(mesh_1, C_gauss_idxs, C_gauss, k=10)
full_C_mean_1 = analysis.interpolate_on_mesh(mesh_1, C_mean_idxs, C_mean, k=10)

# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_1, "gauss_curv_1", header=f"gauss (1/{img_unit}^2)", folderpath=resdata_dir)
datahandler.save_array(full_C_mean_1, "mean_curv_1", header=f"mean (1/{img_unit})", folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_1, title=f"Gauss AVG = {full_C_gauss_1.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, "gauss_hist.png"))
visuals.plot_hist(full_C_mean_1, title=f"Mean AVG = {full_C_mean_1.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, "mean_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_C_gauss_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, "gauss_curv.png"))
visuals.plot_maxproj_pts(verts=mesh_1.vertices, unit=img_unit, colors=full_C_mean_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, "mean_curv.png"))

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_1, mesh_1],
                                   [visuals.color_scalar(full_C_gauss_1, normalise=True, cmap="Spectral"),
                                    visuals.color_scalar(full_C_mean_1, normalise=True, cmap="Spectral")],
                                   name_list=["Gauss", "Mean"])

### |2.1| Curvature Mesh #2

In [ ]:
# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_2, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=np.array(gauss_crop_range),
                                                  mean_crop_range=np.flip(np.array(mean_crop_range) * -1))

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_2 = analysis.interpolate_on_mesh(mesh_2, C_gauss_idxs, C_gauss, k=10)
full_C_mean_2 = analysis.interpolate_on_mesh(mesh_2, C_mean_idxs, C_mean, k=10)
# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_2, "gauss_curv_2", header=f"gauss (1/{img_unit}^2)", folderpath=resdata_dir)
datahandler.save_array(full_C_mean_2, "mean_curv_2", header=f"mean (1/{img_unit})", folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_2, title=f"Gauss AVG = {full_C_gauss_2.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, "gauss_hist.png"))
visuals.plot_hist(full_C_mean_2, title=f"Mean AVG = {full_C_mean_2.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, "mean_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_2.vertices, unit=img_unit, colors=full_C_gauss_2, cmap="Spectral",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, "gauss_curv.png"))
visuals.plot_maxproj_pts(verts=mesh_2.vertices, unit=img_unit, colors=full_C_mean_2, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, "mean_curv.png"))

In [ ]:
# ==== 3D Render Result ====
visuals.view_colored_mesh_multiple([mesh_2, mesh_2],
                                   [visuals.color_scalar(full_C_gauss_2, normalise=True, cmap="Spectral"),
                                    visuals.color_scalar(full_C_mean_2, normalise=True, cmap="Spectral")],
                                   name_list=["Gauss", "Mean"])

### |2.1| Curvature Mesh Comparative View

In [ ]:
# ==== 3D Render Result ====
cmap_lims_render_gauss = [np.min([full_C_mean_1.min(), full_C_mean_2.min()]),
                          np.max([full_C_mean_1.max(), full_C_mean_2.max()])]
cmap_lims_render_mean = [np.min([full_C_gauss_1.min(), full_C_gauss_2.min()]),
                         np.max([full_C_gauss_1.max(), full_C_gauss_2.max()])]
visuals.view_colored_mesh_multiple([mesh_1, mesh_2, mesh_1, mesh_2],
                                   [visuals.color_scalar(full_C_mean_1, cmap="Spectral",
                                                         manual_vminmax=cmap_lims_render_gauss),
                                    visuals.color_scalar(full_C_mean_2, cmap="Spectral",
                                                         manual_vminmax=cmap_lims_render_gauss),
                                    visuals.color_scalar(full_C_gauss_1, cmap="coolwarm",
                                                         manual_vminmax=cmap_lims_render_mean),
                                    visuals.color_scalar(full_C_gauss_2, cmap="coolwarm",
                                                         manual_vminmax=cmap_lims_render_mean)],
                                   name_list=["Mean Mesh 1", "Mean Mesh 2", "Gauss Mesh 1", "Gauss Mesh 2"])

## |3| Single Mesh Gaussian & Mean Curvatures

In [ ]:
mesh_curv_name = "sampling_mesh.ply"
mesh_curv = datahandler.load_mesh(os.path.join(resdata_dir, mesh_curv_name), recalc_normals=True, clean=False)
visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, meshes=[mesh_curv], normal_vecfreq=5, normal_scale=0.05,
                 slice_depth=1, show_mesh_normals=True)

# ==== Define Crop Range of Gauss & Mean Curvature ====
gauss_exp = 1 / (np.ptp(mesh_curv.vertices, axis=0).mean() / 2) ** 2
mean_exp = 1 / (np.ptp(mesh_curv.vertices, axis=0).mean() / 2)
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2), "
      f"Mean should be around {mean_exp:.2e} (1/{img_unit})")
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
mean_order_min, mean_order_max = round(np.log10(mean_exp)) - 1, round(np.log10(mean_exp)) + 1
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]
mean_crop_range = [0.01 * 10 ** mean_order_min, 10 ** mean_order_max]

# gauss_crop_range, mean_crop_range = None, None

# ==== Define Number of Random Calculation Selection ====
curvature_num_samples = 3000

# ==== Define Number of Nearest Neighbours to use for surface fit ====
curvature_patch_info = ["nearest", 20]
# curvature_patch_info = ["radius", 100]
# ==== [Optional] Exclude Boundary Vertices ====
curvature_filter_boundary = False
curvature_filter_boundary_factor = 0.0  # exclude strength between 0 (max) and 1 (no exclusion)

# ==== Calculate Gauss & Mean Curvature ====
curvature_results = analysis.curvature_by_srf_fit(mesh_curv, num_sample=curvature_num_samples,
                                                  patch_mode=curvature_patch_info[0],
                                                  patch_size=curvature_patch_info[1],
                                                  debug=True, filter_boundary=curvature_filter_boundary,
                                                  boundary_excl_factor=curvature_filter_boundary_factor,
                                                  gauss_crop_range=gauss_crop_range, mean_crop_range=mean_crop_range)

C_gauss, C_mean, C_gauss_idxs, C_mean_idxs = curvature_results
full_C_gauss_1 = analysis.interpolate_on_mesh(mesh_curv, C_gauss_idxs, C_gauss, k=10)
full_C_mean_1 = analysis.interpolate_on_mesh(mesh_curv, C_mean_idxs, C_mean, k=10)

# ==== Save Gauss & Mean Curvature ====
datahandler.save_array(full_C_gauss_1, f"{mesh_curv_name}_gauss_curv", header=f"gauss (1/{img_unit}^2)",
                       folderpath=resdata_dir)
datahandler.save_array(full_C_mean_1, f"{mesh_curv_name}_mean_curv", header=f"mean (1/{img_unit})",
                       folderpath=resdata_dir)

# ==== Plot Gauss & Mean Curvature ====
visuals.plot_hist(full_C_gauss_1, title=f"Gauss AVG = {full_C_gauss_1.mean():.2e} $(1/{img_unit}^2)$",
                  savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_gauss_hist.png"))
visuals.plot_hist(full_C_mean_1, title=f"Mean AVG = {full_C_mean_1.mean():.2e} $(1/{img_unit})$",
                  savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_mean_hist.png"))
visuals.plot_maxproj_pts(verts=mesh_curv.vertices, unit=img_unit, colors=full_C_gauss_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Gaussian Curvature $(1/{img_unit}^2)$",
                         savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_gauss_curv.png"))
visuals.plot_maxproj_pts(verts=mesh_curv.vertices, unit=img_unit, colors=full_C_mean_1, cmap="Spectral",
                         hexsize=100, cmap_label=f"Mean Curvature $(1/{img_unit})$",
                         savefig=os.path.join(resfig_dir, f"{mesh_curv_name}_mean_curv.png"))

# Projection onto Mesh

## |1| Global Projection

In [ ]:
# ==== Define Projection Range ====
dist_min = 0
dist_max = 50
dist_num = int(abs(dist_max - dist_min) / np.round(np.min(img_scale) * 2, 2))
proj_mode = "mean"  # "max" or "mean"

# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])

# ==== Project onto Mesh ====
proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=dist_min_custom,
                                scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num,
                                mode=proj_mode, show_proj=True,
                                savefig=os.path.join(resfig_dir, "distgraph_broad-scan.png"),
                                normalise=False)

# ==== Plot Projected Result ====
# visuals.plot_maxproj_pts(verts=sampl_mesh.vertices, colors=proj_broad, cmap="Greens", hexsize=200,
#                          savefig=os.path.join(resfig_dir, "maxproj_broad-scan.png"), unit=img_unit, figsize=(18, 8))

# ==== Plot Histogram of Projected Values ====
visuals.plot_hist(proj_broad, title="Intensities (a.u.)")

In [ ]:
# ==== 3D Render Projected Result on Scaled Mesh ====
# sampl_mesh_layer = analysis.scale_mesh(mesh=sampl_mesh, distance=dist_max)
sampl_mesh_layer = sampl_mesh.copy()
visuals.view_colored_mesh_multiple(mesh_list=[sampl_mesh_layer],
                                   vert_colors_list=[
                                       visuals.color_scalar(proj_broad / proj_broad.max(), cmap="inferno")],
                                   mesh_blending_list=["opaque"])  #, img=img_raw, scale=img_scale)

# ==== 3D Render Projected Result ====
# visuals.view_colored_mesh(mesh=sampl_mesh,
#                           vert_colors=visuals.color_scalar(proj_broad / proj_broad.max(), cmap="inferno"),  #Greens_r
#                           mesh_blending="opaque")  #, img=img_raw, scale=img_scale)

In [ ]:
# ==== Plot Spherical Projection ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices)  #, ref_point=[sphere_x0, sphere_y0, sphere_z0],rotate=[90, 0, 90])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_broad, ptview=False,
                                  hexgridsize=200, cmap="inferno",
                                  savefig=os.path.join(resfig_dir, "distgraph_broad-scan_spherical-projection.png"),
                                  figsize=(14, 8), aspect="equal")

## |2| Onion-peeled view

In [ ]:
# ==== Define Projection Range ====
depths = np.array([10, 15, 20, 25, 30, 35, 40, 45, 50])
# depths = np.array([25, 26, 27, 28, 29, 30])  # HYDRA
depths = np.array([20, 21, 22, 23, 24])  # HYDRA
proj_mode = "mean"
layer_thickness = np.round(np.min(img_scale) / 4, 2)
all_min = depths - layer_thickness
all_max = depths + layer_thickness
print(f"Projection depths: {depths} {img_unit}")
all_num = [20 for i in range(len(all_min))]

all_proj = []
all_mesh = []
all_names = [f"{depths[i]}±{layer_thickness}{img_unit}" for i in range(len(depths))]

for i in range(len(depths)):
    print(f">> Projecting at depth {all_names[i]} !")
    proj_broad = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                    scale=img_scale, min_dist=all_min[i], max_dist=all_max[i], num_dist=all_num[i],
                                    mode=proj_mode, show_proj=False, normalise=False)
    sampl_mesh_layer = analysis.scale_mesh(mesh=sampl_mesh, distance=depths[i])
    # sampl_mesh_layer = sampl_mesh.copy()
    all_proj.append(proj_broad)
    all_mesh.append(sampl_mesh_layer)
all_blendings = ["opaque" for i in range(len(depths))]
all_cmaps = ["inferno" for _ in all_proj]
all_colors = [visuals.color_scalar(proj_iter / proj_iter.max(), cmap=all_cmaps[i]) for i, proj_iter in
              enumerate(all_proj)]

In [ ]:
visuals.view_colored_mesh_multiple(mesh_list=all_mesh, vert_colors_list=all_colors, mesh_blending_list=all_blendings,
                                   name_list=all_names)  #, img=img_raw, scale=img_scale)

### |2.1| EMBL Spherical Projection

In [ ]:
sphere_x0, sphere_y0, sphere_z0, sphere_radius = datahandler.load_array("sphere_fit", folderpath=resdata_dir)[0, :]
print(f"sphere_x0 = {sphere_x0} {img_unit}")
print(f"sphere_y0 = {sphere_y0} {img_unit}")
print(f"sphere_z0 = {sphere_z0} {img_unit}")
print(f"sphere_radius = {sphere_radius} {img_unit}")

In [ ]:
min_proj_dist = 2.0
max_proj_dist = 6.0
slice_proj = 0.5
num_proj_samples = int(np.max(img_scale) / np.min(img_scale))
proj_mode = "mean"
min_all = np.arange(min_proj_dist, max_proj_dist, slice_proj)
max_all = min_all + slice_proj
mask = max_all <= max_proj_dist
min_all, max_all = min_all[mask], max_all[mask]
proj_tasks = np.stack([
    min_all,
    max_all,
    np.full(min_all.shape, num_proj_samples)
], axis=1)

proj_radii = sphere_radius - (min_all + 0.5 * slice_proj)
print(f">> Projecting at radii {proj_radii} {img_unit} ...")
# Flip direction of projection for going "inwards"
proj_tasks[:, :2] *= -1

all_projections = np.empty((len(proj_tasks), len(sampl_mesh.vertices)))
for i, proj_task in enumerate(proj_tasks):
    dist_min, dist_max, dist_num = proj_task
    all_projections[i] = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, unit=img_unit, min_dist_per_vert=None,
                                            scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num,
                                            mode=proj_mode, show_proj=False, savefig="", normalise=False)
print("=======")
print(f"Projected {len(all_projections)} layers !")

In [ ]:
proj_xcords, proj_ycords = analysis.spherical_project(pts=sampl_mesh.vertices,
                                                      ref_point=[sphere_x0, sphere_y0, sphere_z0])
radial_projection = analysis.create_radial_stack(values=all_projections,
                                                 phi_coords=proj_xcords,
                                                 theta_cords=proj_ycords,
                                                 grid_n=np.mean(img_dim[1:]), projection_radii=proj_radii)
radial_stack, stack_cords = radial_projection
mid_radial_stack_i = radial_stack.shape[0] // 2

visuals.plot_matrix(radial_stack[mid_radial_stack_i], figsize=(14, 8), origin="upper",
                    title=f"R = {proj_radii[mid_radial_stack_i]:,.2f} {img_unit}",
                    savefig=os.path.join(resfig_dir, f"radial-stack_{mid_radial_stack_i}.png"),
                    unit="px", colorbar=True, cmap="inferno")

np.savez_compressed(os.path.join(resdata_dir, "radial_projection.npz"), radial_stack=radial_stack,
                    stack_cords=stack_cords)

In [ ]:
radial_stack = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["radial_stack"]
stack_cords = np.load(os.path.join(resdata_dir, "radial_projection.npz"))["stack_cords"]

datahandler.save_tiff(radial_stack, filepath=os.path.join(resfig_dir, "radial-stack.tiff"))

In [ ]:
visuals.view_colored_mesh_multiple([sampl_mesh, sampl_mesh], vert_colors_list=[
    visuals.color_scalar(analysis.normalise_range(all_projections[0]), cmap="Greens_r"),
    visuals.color_scalar(analysis.normalise_range(all_projections[2]), cmap="Greens_r")])

In [ ]:
# stack_coords_phi = stack_cords[0, ..., 0]
# visuals.plot_matrix(stack_coords_phi, origin="upper", colorbar=True, cmap="coolwarm")
# stack_coords_theta = stack_cords[0, ..., 1]
# visuals.plot_matrix(stack_coords_theta, origin="upper", colorbar=True, cmap="coolwarm")
# stack_coords_radii = stack_cords[:, 0, 0, 2]
# plt.figure()
# plt.hexbin(proj_ycords, proj_xcords, all_projections[0], gridsize=100)
# plt.gca().invert_yaxis()
# plt.gca().invert_xaxis()
# plt.show()

In [ ]:
# ==== Experimental: Spherical Projection relative to Sphere Fit ====
sphere_params_load = datahandler.load_array("sphere_fit", folderpath=resdata_dir)
sphere_x0, sphere_y0, sphere_z0, sphere_radius = sphere_params_load[0, :]
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices, ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, cmap="Greens_r", invert_y_axis=True,
                                  intensities=proj_broad, figsize=(14, 8), ptview=True, aspect="equal",
                                  savefig=os.path.join(resfig_dir, "distgraph_broad-scan_spherical-projection.png"),
                                  ptsize=0.1)
# ==== Experimental: Different Angle Spherical Projection relative to Sphere Fit ====
rotation_angles = [[0, 0, 0],
                   [np.pi / 4, 0, 0],
                   [np.pi / 2, 0, 0],
                   [np.pi / 2, 0, np.pi / 2]]
for rot_angles in rotation_angles:
    print(f"Rotation angles: {np.degrees(rot_angles)}")
    sph_proj_phi, sph_proj_theta = analysis.spherical_project(
        pts=sampl_mesh.vertices, rotate=rot_angles, ref_point=[sphere_x0, sphere_y0, sphere_z0])
    visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_broad,
                                      aspect="equal", ptview=True, figsize=(14, 8))

### |2.2| Check Multi-Layering

In [ ]:
# ==== Define Projection Range and Intermediate Value ====
# dist_min = 0
# dist_max = 30
dist_middle = dist_min + (dist_max - dist_min) / 2
dist_num = 20
print(f"Chosen Middle Distance {dist_middle} {img_unit}")
# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])

# ==== Project onto Mesh ====
proj_full = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, min_dist_per_vert=dist_min_custom,
                               scale=img_scale, min_dist=dist_min, max_dist=dist_max, num_dist=dist_num, mode="max",
                               show_proj=False, return_full=True, unit=img_unit, normalise=False)
# ==== Plot Projected Result ====
distances, dist_points, radial_intensities = proj_full
distcolor, distcmap = visuals.colour_dist(distances=distances, middle_val=dist_max - dist_middle,
                                          radial_points=dist_points, mesh=sampl_mesh,
                                          radial_intensities=radial_intensities)
# visuals.plot_maxproj_pts(verts=sampl_mesh.vertices, colors=distcolor, unit=img_unit, cmap=distcmap, hexsize=300,
#                          savefig=os.path.join(resfig_dir, "maxproj_multi-layer.png"))

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=sampl_mesh, color_override=distcolor)  #, img=img_raw, scale=img_scale)

In [ ]:
# EXPERIMENTAL Ikmi
visuals.view_colored_mesh(mesh=sampl_mesh,
                          color_override=analysis.colour_dist_test(distances=distances, radial_points=dist_points,
                                                                   mesh=sampl_mesh,
                                                                   radial_intensities=radial_intensities))

In [ ]:
# ==== Plot Spherical Projection ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=sampl_mesh.vertices)  #,ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=distcolor, ptview=True,
                                  savefig=os.path.join(resfig_dir, "multi-layer_spherical-projection.png"),
                                  cmap=distcmap,
                                  figsize=(14, 8))

In [ ]:
# ==== Plot Spherical Projections ====
rotation_angles = [[0, 0, 0],
                   [np.pi / 4, 0, 0],
                   [np.pi / 2, 0, 0],
                   [np.pi / 2, 0, np.pi / 2]]
for rot_angles in rotation_angles:
    print(f"Rotation angles: {np.degrees(rot_angles)}")
    sph_proj_phi, sph_proj_theta = analysis.spherical_project(
        pts=sampl_mesh.vertices, rotate=rot_angles)
    visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=distcolor, ptview=True,
                                      figsize=(5, 5), cmap=distcmap)

## |3| Isolate single layer

In [ ]:
# ==== Projection Logic ====
layer_depth = 25
layer_thickness = np.round(np.min(img_scale) / 2, 2)
dist_min = layer_depth - layer_thickness / 2
dist_max = layer_depth + layer_thickness / 2
proj_mode = "mean"
layer_label = f"proj_{dist_min}_to_{dist_max}_{img_unit}"
dist_middle = dist_min + (dist_max - dist_min) / 2

# ==== Specify Custom Minimum ====
dist_min_custom = None
# dist_min_custom = np.linspace(0, 5, sampl_mesh.vertices.shape[0])
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
if not os.path.exists(resdata_dir_layer):
    os.makedirs(resdata_dir_layer)
if not os.path.exists(resfig_dir_layer):
    os.makedirs(resfig_dir_layer)

# ==== Save Sampling Vertices and Normals ====
datahandler.save_array(sampl_mesh.vertices, "verts", header="x,y,z", folderpath=resdata_dir_layer)
datahandler.save_array(sampl_mesh.vertex_normals, "normals", header="nx,ny,nz", folderpath=resdata_dir_layer)
dist_num = 30

# ==== Project onto Mesh ====
proj_layer = analysis.proj2mesh(img=img_raw, mesh=sampl_mesh, min_dist_per_vert=dist_min_custom,
                                scale=img_scale, min_dist=dist_min, max_dist=dist_max,
                                num_dist=dist_num, mode=proj_mode, show_proj=True,
                                savefig=os.path.join(resfig_dir_layer, f"distgraph.png"), unit=img_unit,
                                normalise=False)

# ==== Save Projection ====
layer_mesh = analysis.scale_mesh(mesh=sampl_mesh, distance=dist_middle)
# layer_mesh = sampl_mesh.copy()
datahandler.save_mesh(layer_mesh, filepath=os.path.join(resdata_dir_layer, "layer_mesh.ply"))
datahandler.save_array(proj_layer, "intensities", header="I", folderpath=resdata_dir_layer)

# ==== Plot Projected Result ====
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_layer, hexgridsize=400,
                                  savefig=os.path.join(resfig_dir_layer, "spherical-projection.png"), cmap="Greens_r")

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                            cmap="Greens_r"))  #, img=img_raw, scale=img_scale)

# -- Load Layer --

In [ ]:
layer_names = [
    d for d in os.listdir(resdata_dir)
    if os.path.isdir(os.path.join(resdata_dir, d))
]
print(f"Found layer(s): {layer_names}")

In [ ]:
# ==== Load Projected Result ====
# layer_label = f"proj_{dist_min}_to_{dist_max}_{img_unit}"
# layer_label = 'proj_2.0_to_6.0_um_outer'
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
proj_layer /= proj_layer.max()
try:
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir_layer, "layer_mesh.ply"), recalc_normals=False,
                                       clean=False)
except:
    print("Could not find layer mesh, using smampling mesh instead..")
    layer_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True, clean=False)

# ==== Plot Projected Result ====
visuals.plot_hist(proj_layer, title="Intensities (a.u.)")
sph_proj_phi, sph_proj_theta = analysis.spherical_project(
    pts=layer_mesh.vertices)  #, ref_point=[sphere_x0, sphere_y0, sphere_z0])
visuals.plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=proj_layer, hexgridsize=400,
                                  cmap="Greens_r")

In [ ]:
# ==== 3D Render Projected Result ====
visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                            cmap="Greens_r"))  #img=img_raw,scale=img_scale

# -- Tangential Nematic Analysis --

### |1.1| Select and Define surface patches

In [ ]:
# ==== Select Patch Type and Size ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 1500]
compute_num = 3000

# ==== Pres-Select Vertices for 2D+ Orientation Analysis ====
idxs_sel = np.arange(layer_mesh.vertices.shape[0])
idxs_sel = analysis.filter_normal_validity(mesh=layer_mesh, idxs_sel=idxs_sel, k=20, threshold=0.99)

# ==== Filter by Intensity Value ====
cutoff_min_intensity = 0.0 * np.max(proj_layer)
cutoff_max_intensity = 1.0 * np.max(proj_layer)
idxs_sel = idxs_sel[(proj_layer[idxs_sel] > cutoff_min_intensity) & (proj_layer[idxs_sel] < cutoff_max_intensity)]

# ==== Compute only points at Interval ====compute_num
idxs_sel = np.random.choice(idxs_sel, size=int(compute_num))
if len(idxs_sel) == 0:
    print("!! ERROR: No vertices were selected for analysis !!")

# ==== Search Nearest Neighbours ====
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
if patch_type == "radius":
    idxs_neigh = analysis.coord_search_radius(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                              r=patch_size)
    title_render = f"Extracted directors w.r.t {patch_size}{img_unit}"
elif patch_type == "nearest":
    idxs_neigh = analysis.coord_search_neighbours(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                                  k=patch_size, n_process=8)
    title_render = f"Extracted directors w.r.t {patch_size - 1} neighbours"
else:
    idxs_neigh = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# # ==== Filter Valid Surface Patches ====
# idxs_neigh, valid_patch_idxs = analysis.filter_valid_patches(verts=layer_mesh.vertices, idxs_neigh=idxs_neigh,
#                                                              factor=0.1)
# idxs_sel = idxs_sel[valid_patch_idxs]

# # ==== Filter by Intensity Variation ====
# cutoff_intensity_variance = 0.0
# proj_layer_variance = np.array([np.var(proj_layer[patch_idxs]) for patch_idxs in idxs_neigh])
# visuals.plot_hist(proj_layer_variance, title="Intensity Variance per Patch")
# filter_intens_var_mask = proj_layer_variance > cutoff_intensity_variance

# Filter idxs_sel
# idxs_sel = idxs_sel[filter_intens_var_mask]

# Filter idxs_neigh (keep only the patches that passed)
# idxs_neigh = [patch for i, patch in enumerate(idxs_neigh) if filter_intens_var_mask[i]]

# ==== Find Nearest Intensities ====
proj_layer_neigh = [proj_layer[patch_idxs] for patch_idxs in idxs_neigh]

# ==== Save Vertices for 2D+ Orientation Analysis ====
print(f"Num of directors to be calculated: {len(idxs_sel)} !")
datahandler.save_array(idxs_sel, "calcindeces", header="idx", folderpath=resdata_dir_layer)

# ==== Create the Tangential Bases ====
neighbors_coords = [layer_mesh.vertices[patch] for patch in idxs_neigh]
central_normals = layer_mesh.vertex_normals[idxs_sel]

tan_cords, tan_x, tan_y = analysis.tan_proj(neighbors_coords, central_normals)

# ==== Save the Tangential Bases ====
datahandler.save_array(tan_x, "tan_x", header="t1x,t1y,t1z", folderpath=resdata_dir_layer)
datahandler.save_array(tan_y, "tan_y", header="t2x,t2y,t2z", folderpath=resdata_dir_layer)

In [ ]:
# ==== 3D Render Vertices for 2D+ Orientation Analysis ====
visuals.view_colored_verts(verts=layer_mesh.vertices[idxs_sel],
                           colors=visuals.color_scalar(proj_layer[idxs_sel], cmap="Greens_r"), scale=img_scale)

In [ ]:
# # ==== 3D Render all vertices to be calculated on ====
# all_calculated_mask = np.isin(np.arange(layer_mesh.vertices.shape[0]), idxs_sel)
# all_calculated_color = ["red" if i else "grey" for i in all_calculated_mask]
# visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=all_calculated_color)  #, img=img_raw, scale=img_scale)

### |1.2| Extract Directors

In [ ]:
# ==== Tune Local Orientation Extraction Accuracy ====
grid_N = 30
box_size = 10
debug_2dcurve_analysis = False
debug_vert_idx, debug_grid_x, debug_grid_y, debug_grid_z = None, None, None, None

print(f">> Using local grid of NxN: {grid_N} and box size: {box_size}...")

if debug_2dcurve_analysis:
    # ==== Pick single vertex for debug ====
    debug_vert_idx = np.random.choice(np.arange(idxs_sel.shape[0]))
    big_grid = analysis.tan_interp_batch(
        coords=[tan_cords[debug_vert_idx]],
        intensities=[proj_layer_neigh[debug_vert_idx]],
        grid_size=grid_N
    )[2][0]
    print(big_grid.shape)
else:
    big_grid = np.vstack([grid.T for grid in analysis.tan_interp_batch(
        coords=tan_cords,
        intensities=proj_layer_neigh,
        grid_size=grid_N
    )[2]])

# ==== Extract Directors ====
directors_2dcurved = analysis.batch_2d_orientation(
    big_grid=big_grid, box_size=box_size,
    vertices=layer_mesh.vertices[idxs_sel],
    tan_x=tan_x, tan_y=tan_y,
    debug=debug_2dcurve_analysis,
    debug_idx=debug_vert_idx,
    debug_line_length=1
)

if not debug_2dcurve_analysis:
    # ==== Save Directors ====
    datahandler.save_array(directors_2dcurved, "directors_2dcurved", header="x,y,z,vx,vy,vz",
                           folderpath=resdata_dir_layer)

    # ==== Plot Directors ====
    visuals.plot_dir_field(directors=directors_2dcurved, title=title_render,
                           savefig=os.path.join(resfig_dir_layer, "directors_2dcurved.png"), veclength=10)

In [ ]:
# ==== 3D Render Directors ====
veclength = 10
vecwidth = 0.5
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              verts=layer_mesh.vertices, verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"),
#                              edge_width=veclength / 6, length=veclength, vec_opacity=0.5, pts_size=1, pts_opacity=0.8,
#                              img=None, scale=img_scale)

# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              edge_width=veclength / 6, length=veclength)

# visuals.view_mesh_dir_field([layer_mesh], directors=directors_2dcurved, vec_colors="red",
#                             vec_edge_width=veclength / 6, vec_length=veclength)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"), vec_length=veclength,
                                    vec_edge_width=vecwidth)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10),
    savefig=os.path.join(resfig_dir_layer, "spherical_projection_extracted-directors.png")
)

### -- Load Directors --

In [ ]:
# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10)
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, vec_edge_width=0.2,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"))

### |2.1| Remove Initial Noise by Nematic Averaging

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours or Radius ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 10
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)
veccoords = directors_2dcurved[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order-init_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg-init_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_intial-avg-nematic_{patch_label}.png")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_intial-order-s_{patch_label}.png")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])
directors_2dcurved_avg_init = directors_2dcurved_avg.copy()

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 0.5
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=0.5 * vec_length, vec_edge_width=vec_edge_width)

# visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
#                                     vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
#                                     mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
#                                                                           cmap="Greys_r"),
#                                     vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)

# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"))


In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

### |2.2| Compute nematic order scalar S

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)

veccoords = directors_2dcurved_avg_init[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved_avg_init,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved_avg_init.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.png")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.png")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=5,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    savefig=os.path.join(resfig_dir_layer, f"spherical_projection_field_avg-nematic_{patch_label}.png"),
)

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 3
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)
#
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(analysis.normalise_range(proj_layer), cmap="Greens_r"))


### -- Load 2D+ Order --

In [ ]:
# ==== Load 2D+ nematic order ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

load_label_inset = "-init"
directors_2dcurved_avg = datahandler.load_array(f"directors-avg{load_label_inset}_2dcurved_{patch_label}",
                                                folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order{load_label_inset}_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
visuals.plot_hist(S_2dcurv, title=title_hist, xlim=[0, 1])
visuals.plot_dir_field(directors=directors_2dcurved_avg, veccolor=S_2dcurv, veclength=4, view_init=(20, 0),
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1], show_axes=False)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

In [ ]:
vec_length = 10
vec_edge_width = 3
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=0.2)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"), vec_length=vec_length,
                                    vec_edge_width=vec_edge_width)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh(layer_mesh,
                          visuals.color_scalar(analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=10),
                                               manual_vminmax=[0, 1]))

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_multiple([layer_mesh, layer_mesh],
                                   [visuals.color_scalar(
                                       analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=30),
                                       manual_vminmax=[0, 1]),
                                       visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r")])

### |3| Identify Defect Location(s) + Geodesic Distance

In [ ]:
# ==== Find Defects and Inter ====
dist_cutoff_defect_localisation = 50
max_candidates_defect_localisation = 10
defect_idxs, rel_dists = analysis.select_geodesic_defects(S_2dcurv, layer_mesh, idxs_sel,
                                                          dist_cutoff=dist_cutoff_defect_localisation, unit=img_unit,
                                                          max_candidates=max_candidates_defect_localisation)
datahandler.save_array(defect_idxs, name="defect-idxs", header="idx", folderpath=resdata_dir_layer)

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1],
                       savefig=os.path.join(resfig_dir_layer, f"defect-locations.png"))

In [ ]:

sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
    savefig=os.path.join(resfig_dir_layer, f"defect-locations.png")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, vec_edge_width=0.1,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

In [ ]:
# ==== Calculate Geodesic Distance between Defects ====
defect_1_index = idxs_sel[defect_idxs[0]]
defect_2_index = idxs_sel[defect_idxs[1]]
dist = analysis.geodesic_distmesh(mesh=layer_mesh, index1=defect_1_index, index2=defect_2_index, debug=True)
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.3,
                       veccolor="red", marker=layer_mesh.vertices[[defect_1_index, defect_2_index]],
                       pt_label="Points", pt_alpha=1.0, view_init=[90, 0])

### -- Load Defect(s) Position(s) --

In [ ]:
defect_idxs = datahandler.load_array(name="defect-idxs", folderpath=resdata_dir_layer).astype(int)
print(f"Found {len(defect_idxs)} defect indeces {defect_idxs}!")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

### |4| Topological Charge of Nematic Point Defects

In [ ]:
# # ==== Calculate Gaussian Curvature for Topological Charge Analysis ====
gauss_exp = 1 / (np.ptp(layer_mesh.vertices, axis=0).mean() / 2) ** 2
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2)")
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]

patch_info = ["radius", 30]
# patch_info = ["nearest", 20]
# gauss_crop_range = None
curv_charge_quick = analysis.curvature_by_srf_fit(mesh=layer_mesh, patch_mode=patch_info[0], patch_size=patch_info[1],
                                                  debug=True, gauss_crop_range=gauss_crop_range, num_sample=1000)
# Results given as curv_charge_quick = C_gauss, C_mean, Gauss_idxs, mean_idxs
gauss_curv_smooth = analysis.interpolate_on_mesh(mesh=layer_mesh, value_idxs=curv_charge_quick[2],
                                                 values=curv_charge_quick[0], k=3)

In [ ]:
visuals.view_colored_mesh(layer_mesh, visuals.color_scalar(gauss_curv_smooth, normalise=True))

In [ ]:
# ==== Calculate Curved Topological Charge ====
patch_charge = ["radius", 50]
# patch_charge = ["nearest", 1000]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    charge_patch_idxs = analysis.coord_search_radius(layer_mesh.vertices,
                                                     custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                     r=patch_size)
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    charge_patch_idxs = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                         custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                         k=patch_size)
else:
    charge_patch_idxs = None
    print(f"[!] Unknown patch type: {patch_type}")

# charge_patch_idxs, _ = analysis.filter_valid_patches(layer_mesh.vertices, charge_patch_idxs, factor=0.2)
# charge_patch_idxs = analysis.unique_neighborhoods(charge_patch_idxs)
defect_idxs_calc = [np.flatnonzero(idxs_sel == lst[0])[0]
                    for lst in charge_patch_idxs
                    if np.any(idxs_sel == lst[0])]

_, tan_x_all, tan_y_all = analysis.tan_proj(layer_mesh.vertices[:, np.newaxis], layer_mesh.vertex_normals)
m_charge, calc_charge_loop_idxs = analysis.curved_nem_charge(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                                             calc_idxs=defect_idxs_calc,
                                                             director_indeces=idxs_sel,
                                                             tan_x=tan_x_all, tan_y=tan_y_all,
                                                             c_gauss=gauss_curv_smooth,
                                                             loop_angle_precision=1, patch_mode=patch_type,
                                                             patch_size=patch_size, debug=True,
                                                             correct_orientation=True)
print(f"Final number of defects: {len(m_charge)} !")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.02, veccolor=S_2dcurv,
                       marker=np.vstack([layer_mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line",
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

m_charge_extended = np.full(len(layer_mesh.vertices), 0.0)
if patch_type == "radius":
    charge_patch_idxs_calc = analysis.coord_search_radius(layer_mesh.vertices,
                                                          custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                                          r=patch_size)
elif patch_type == "nearest":
    charge_patch_idxs_calc = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                              custom_probes=layer_mesh.vertices[
                                                                  idxs_sel[defect_idxs_calc]],
                                                              k=patch_size)
else:
    charge_patch_idxs_calc = None
    print(f"[!] Unknown patch type: {patch_type}")
if isinstance(charge_patch_idxs_calc, list):
    flat_idxs = np.concatenate([np.atleast_1d(np.array(x)) for x in charge_patch_idxs_calc if len(x) > 0])
else:
    flat_idxs = np.ravel(charge_patch_idxs_calc)

m_charge_extended[flat_idxs] = np.repeat(m_charge, [len(np.atleast_1d(x)) for x in charge_patch_idxs_calc])
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=10, veccolor=m_charge_extended[idxs_sel],
                       cmap="rainbow",
                       title=r"TOTAL CHARGE$\approx$" + f"{np.nansum(m_charge):.3}",
                       cmap_label="topological charge $m$",
                       manual_vminmax=[-1, 1], show_axes=False, marker=layer_mesh.vertices[idxs_sel[defect_idxs_calc]])

# ==== Save Curved Topological Charge ====
datahandler.save_array(m_charge, name=f"top-charge_2dcurved_{patch_label}", header="m", folderpath=resdata_dir_layer)
datahandler.save_array(defect_idxs_calc, name=f"top-charge_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)

# ==== Plot Curved Topological Charge ====
visuals.plot_hist(m_charge, title=f"Sum(m)={np.nansum(m_charge)}",
                  savefig=os.path.join(resfig_dir_layer, f"hist_top-charge_{patch_label}"))



In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(m_charge_extended[idxs_sel], manual_vminmax=[-1, 1],
                                                                    cmap="rainbow"), vec_edge_width=0.05,
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.2,
                                    vec_length=10,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.png")
)

### -- Load Defect(s) Charge(s) --

In [ ]:
patch_charge = ["radius", 40]
# patch_charge = ["nearest", 200]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

m_charge = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
defect_idxs_calc = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}_idxs",
                                          folderpath=resdata_dir_layer).astype(int)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.png")
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greens_r"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]], vec_edge_width=0.2,
                                    vec_length=10,
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

### (experimental) Defect Polarisation

In [ ]:
patch_polarisation = ["radius", 40]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")
pol_vecfield, pol_idxs = extra_nematic.compute_defect_polarisations(
    mesh=layer_mesh,
    idxs_sel=idxs_sel,
    directors=directors_2dcurved_avg,
    vertex_normals=layer_mesh.vertex_normals.copy(),
    defect_idxs_calc=defect_idxs_calc,
    m_charge=np.round(m_charge, 2),
    patch_type=patch_type,
    patch_size=patch_size, show_profile=True
)
charge_pol_linked_idxs = np.argsort(defect_idxs_calc)[
    np.searchsorted(defect_idxs_calc, pol_idxs, sorter=np.argsort(defect_idxs_calc))]

datahandler.save_array(pol_vecfield, name=f"def-pol_2dcurved_{patch_label}", header="x,y,z,vx,vy,vz",
                       folderpath=resdata_dir_layer)
datahandler.save_array(charge_pol_linked_idxs, name=f"def-pol_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=10,
                                    marker_vector_width=1, vec_edge_width=0.05,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

### -- Load Defect(s) Polarisation(s) --

In [ ]:
patch_polarisation = ["radius", 40]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

pol_vecfield = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
charge_pol_linked_idxs = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}_idxs",
                                                folderpath=resdata_dir_layer).astype(int)

In [ ]:
rotation_angles = [1, 1, 1]
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices, rotate=rotation_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:], rotate=rotation_angles)
pol_dir_phi, pol_dir_theta = analysis.spherical_project_vectors(pol_vecfield[:, :3], pol_vecfield[:, 3:],
                                                                rotate=rotation_angles)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv, vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(22, 8),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    marker_vec=(pol_dir_phi, pol_dir_theta, idxs_sel[pol_idxs]),
    marker_vec_scale=20, marker_vec_width=0.005, aspect="equal",
    marker_vec_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs], manual_vminmax=[-1, 1],
                                          cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-polarisations.png")
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="inferno"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=50, vec_length=10,
                                    marker_vector_width=7,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

### |5| Criss-Cross

In [ ]:
found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in layer_names
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order_2dcurved") and f.endswith(".csv")
]

print(f"Found layer(s) and patch label(s): {found_analysed_layers}")

(layer_name_1, patch_label_1), (layer_name_2, patch_label_2) = found_analysed_layers[:2]

In [ ]:
try:
    layer_mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, layer_name_1, "layer_mesh.ply"),
                                         recalc_normals=False, clean=False)
except:
    print("Could not find layer mesh, using smampling mesh instead..")
    layer_mesh_1 = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True,
                                         clean=False)

try:
    layer_mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, layer_name_2, "layer_mesh.ply"),
                                         recalc_normals=False, clean=False)
except:
    print("Could not find layer mesh, using smampling mesh instead..")
    layer_mesh_2 = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=True,
                                         clean=False)

In [ ]:
# layer_name_1 = 'proj_4.9_to_5.1_um'
# patch_label_1 = 'r-30um'
# layer_name_2 = 'proj_8.9_to_9.1_um'
# patch_label_2 = 'r-30um'
# (layer_name_1, patch_label_1), (layer_name_2, patch_label_2) = found_analysed_layers[:2]
crisscross_mag, field_1, field_2 = analysis.layers_crisscross(layer_name_1=layer_name_1, patch_label_1=patch_label_1,
                                                              layer_name_2=layer_name_2, patch_label_2=patch_label_2,
                                                              resdata_dir=resdata_dir,
                                                              director_name_prefix="directors-avg_2dcurved_")
datahandler.save_array(crisscross_mag,
                       f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_crisscross_mag",
                       header="strength", folderpath=resdata_dir)

visuals.plot_hist(crisscross_mag, title="Criss-Cross Strength", xlim=[0, 1],
                  savefig=os.path.join(resfig_dir,
                                       f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_hist_crisscross_mag.png"))

visuals.plot_dir_field(directors=field_1, veclength=10, view_init=(20, 0),
                       veccolor=crisscross_mag,
                       cmap_label="Criss-Cross Strength", show_axes=False, manual_vminmax=[0, 1],
                       cmap="coolwarm",
                       savefig=os.path.join(resfig_dir,
                                            f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_nematic-field_crisscross_mag.png"))
rotation_angles = [1, 1, 1]
plotted_vecfields = np.concatenate((field_2, field_1), axis=0)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=plotted_vecfields[:, :3],
                                                          rotate=rotation_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(plotted_vecfields[:, :3],
                                                                plotted_vecfields[:, 3:],
                                                                rotate=rotation_angles)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=np.ones_like(sph_proj_phi),
    vec_pos_phi=sph_proj_phi,
    vec_pos_theta=sph_proj_theta,
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="criss cross magnitude",
    hexgridsize=200,
    scale_factor=5, alpha=0.0,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0),
    vec_manual_vminmax=[0, 1],
    arrow_alpha=0.8, figsize=(22, 8), vec_cmap="coolwarm",
    aspect="equal",
    savefig=os.path.join(resfig_dir,
                         f"{layer_name_1}_{patch_label_1}_VS_{layer_name_2}_{patch_label_2}_nematic-field_crisscross_mag.png")
)


In [ ]:
proj_layer_1 = datahandler.load_array("intensities", folderpath=os.path.join(resdata_dir, layer_name_1))
proj_layer_2 = datahandler.load_array("intensities", folderpath=os.path.join(resdata_dir, layer_name_2))
visuals.view_colored_mesh_multiple(mesh_list=[layer_mesh_1, layer_mesh_2],
                                   vert_colors_list=[visuals.color_scalar(analysis.normalise_range(proj_layer_1),
                                                                          cmap="Greens_r"),
                                                     visuals.color_scalar(analysis.normalise_range(proj_layer_2),
                                                                          cmap="Blues_r")])

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh_1, directors=plotted_vecfields,
                                    vec_colors=visuals.color_scalar(
                                        np.concatenate((np.ones(len(field_2)) * 0.5, crisscross_mag), axis=0),
                                        manual_vminmax=[0, 1], cmap="coolwarm", ),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer_1),
                                                                          cmap="Greys_r"),
                                    vec_edge_width=0.4, vec_length=6)

# (experimental Oriol Gastruloids)

In [ ]:
from scripts import extra_gastruloids

## -- Load Mesh --

In [ ]:
gastr_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=False, clean=False)

## |1| Find 3D Curve

In [ ]:
from batch_analysis.automated_scripts import run_cylindrical_analysis

# img_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas4.tif'
centerline_fitted, mesh_s, mesh_rho, mesh_phi = run_cylindrical_analysis.main(
    img_path=img_path, overwrite=False, show_figures=True, render=False, voxel_size=3, spline_smooth_factor=400)

In [ ]:
resdata_dir, resfig_dir = datahandler.create_resdirs(img_path)
gastr_mesh = datahandler.load_mesh(os.path.join(resdata_dir, "sampling_mesh.ply"), recalc_normals=False,
                                   clean=False)

In [ ]:
visuals.view_colored_mesh(gastr_mesh, vert_colors="white", markers=centerline, mesh_shading="flat",
                          mesh_opacity=0.6, mesh_blending="translucent_no_depth",
                          marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline)), cmap="Blues"))

In [ ]:
visuals.view_colored_mesh_multiple([gastr_mesh, gastr_mesh, gastr_mesh, gastr_mesh],
                                   vert_colors_list=["white",
                                                     visuals.color_scalar(mesh_s,
                                                                          manual_vminmax=[0, mesh_s.max()],
                                                                          cmap="inferno"),
                                                     visuals.color_scalar(mesh_rho,
                                                                          manual_vminmax=[0, mesh_rho.max()],
                                                                          cmap="Spectral"),
                                                     visuals.color_scalar(mesh_phi,
                                                                          manual_vminmax=[-np.pi, np.pi],
                                                                          cmap="hsv")],
                                   markers=centerline_fitted, mesh_shading="flat",
                                   mesh_opacity_list=[0.6, 1.0, 1.0, 1.0],
                                   marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)),
                                                                      cmap="Blues"))

## -- Load 3D Curve --

In [ ]:
from batch_analysis.automated_scripts import run_cylindrical_analysis

# img_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas4.tif'
centerline_fitted, mesh_s, mesh_rho, mesh_phi = run_cylindrical_analysis.main(
    img_path=img_path, overwrite=False, show_figures=False, render=False)

# # ==== 3D Render Curvi-linear Coordinates ====
# visuals.view_colored_mesh_multiple([gastr_mesh, gastr_mesh, gastr_mesh, gastr_mesh],
#                                    vert_colors_list=["white",
#                                                      visuals.color_scalar(mesh_s, manual_vminmax=[0, mesh_s.max()],
#                                                                           cmap="inferno"),
#                                                      visuals.color_scalar(mesh_rho, manual_vminmax=[0, mesh_rho.max()],
#                                                                           cmap="Spectral"),
#                                                      visuals.color_scalar(mesh_phi, manual_vminmax=[-np.pi, np.pi],
#                                                                           cmap="hsv")],
#                                    markers=centerline_fitted, mesh_shading="flat",
#                                    mesh_opacity_list=[0.6, 1.0, 1.0, 1.0],
#                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(centerline_fitted)),
#                                                                       cmap="Blues"))

## -- Load Layer --

In [ ]:
found_projected_layers = [
    d for d in os.listdir(resdata_dir)
    if os.path.isdir(os.path.join(resdata_dir, d))
]

found_analysed_layers = [
    (layer, f.split("_")[-1].split(".csv")[0])
    for layer in found_projected_layers
    for f in os.listdir(os.path.join(resdata_dir, layer))
    if f.startswith("S-order-init_2dcurved") and f.endswith(".csv")
]
print(found_projected_layers, "\n", found_analysed_layers)

In [ ]:
layer_label = 'proj_24.93_to_25.07_um'
patch_avg = ["radius", 30]
# ==== Load Projected Result ====
resdata_dir_layer = os.path.join(resdata_dir, layer_label)
resfig_dir_layer = os.path.join(resfig_dir, layer_label)
proj_layer = datahandler.load_array("intensities", folderpath=resdata_dir_layer)
proj_layer /= proj_layer.max()

# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

# ==== Load 2D+ nematic order ====
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")
directors_2dcurved_avg = datahandler.load_array(f"directors-avg-init_2dcurved_{patch_label}",
                                                folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order-init_2dcurved_{patch_label}", folderpath=resdata_dir_layer)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=gastr_mesh, directors=directors_2dcurved,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors="white")

## |2| Project Layer on 3D Curve

In [ ]:
phi_new_zero = extra_gastruloids.find_phi_intensity_max(phi_vals=mesh_phi, intensity_vals=proj_layer)
mesh_phi = extra_gastruloids.shift_angle_periodic(angle=mesh_phi, angle_zerobase=phi_new_zero)
dir_s, dir_rho, dir_phi = extra_gastruloids.cylindrical_along_curve(points=directors_2dcurved[:, :3],
                                                                    curve=centerline_fitted)
dir_phi = extra_gastruloids.shift_angle_periodic(angle=dir_phi, angle_zerobase=phi_new_zero)

visuals.plot_cylindrical_projection(phi=mesh_phi, rho=mesh_rho, s=mesh_s, colors=proj_layer, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=300, cmap="inferno")

low_cutoff_phi = -np.pi / 3
high_cutoff_phi = np.pi / 3

mesh_s_cropped = extra_gastruloids.crop_by_angles(values=mesh_s, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                  angle_high_cutoff=high_cutoff_phi)
mesh_rho_cropped = extra_gastruloids.crop_by_angles(values=mesh_rho, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                    angle_high_cutoff=high_cutoff_phi)
proj_layer_cropped = extra_gastruloids.crop_by_angles(values=proj_layer, angles=mesh_phi,
                                                      angle_low_cutoff=low_cutoff_phi,
                                                      angle_high_cutoff=high_cutoff_phi)
mesh_phi_cropped = extra_gastruloids.crop_by_angles(values=mesh_phi, angles=mesh_phi, angle_low_cutoff=low_cutoff_phi,
                                                    angle_high_cutoff=high_cutoff_phi)

dir_s_cropped = extra_gastruloids.crop_by_angles(values=dir_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
dir_rho_cropped = extra_gastruloids.crop_by_angles(values=dir_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                   angle_high_cutoff=high_cutoff_phi)
dir_phi_cropped = extra_gastruloids.crop_by_angles(values=dir_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                   angle_high_cutoff=high_cutoff_phi)
visuals.plot_scatter(x=mesh_phi_cropped, y=proj_layer_cropped, title="Projection Intensity vs. Angle",
                     xlabel=r"$\phi$ (rad)", vert_line=[low_cutoff_phi, high_cutoff_phi],
                     ylabel="Projection Intensity (a.u.)", xlim=[-np.pi, np.pi])
visuals.plot_cylindrical_projection(phi=mesh_phi_cropped, rho=mesh_rho_cropped, s=mesh_s_cropped,
                                    colors=proj_layer_cropped, aspect="equal",
                                    title=f"Projected Intensities \n{layer_label}", hexsize=400, figsize=(10, 5),
                                    cmap="inferno")
visuals.plot_rho_profile(mesh_s=mesh_s_cropped, mesh_rho=mesh_rho_cropped, mesh_phi=mesh_phi_cropped, img_unit=img_unit)



## |3| Decompose Nematic Field

In [ ]:
e_s, e_phi, e_rho = extra_gastruloids.create_s_phi_basis(points=directors_2dcurved[:, :3], curve=centerline_fitted,
                                                         normals=gastr_mesh.vertex_normals[idxs_sel])
e_s_cropped = extra_gastruloids.crop_by_angles(values=e_s, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                               angle_high_cutoff=high_cutoff_phi)
e_phi_cropped = extra_gastruloids.crop_by_angles(values=e_phi, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
e_rho_cropped = extra_gastruloids.crop_by_angles(values=e_rho, angles=dir_phi, angle_low_cutoff=low_cutoff_phi,
                                                 angle_high_cutoff=high_cutoff_phi)
directors_2dcurved_cropped = extra_gastruloids.crop_by_angles(values=directors_2dcurved, angles=dir_phi,
                                                              angle_low_cutoff=low_cutoff_phi,
                                                              angle_high_cutoff=high_cutoff_phi)

In [ ]:
# e_s_mesh, e_phi_mesh, e_rho_mesh = extra_gastruloids.create_s_phi_basis(points=gastr_mesh.vertices,
#                                                                         curve=centerline_fitted,
#                                                                         normals=gastr_mesh.vertex_normals)
# freq = 50
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[gastr_mesh.vertices[::freq], gastr_mesh.vertices[::freq]],
#     vec_dir=[e_s_mesh[::freq], e_phi_mesh[::freq]], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])

In [ ]:
# directors_2dcurved_cropped[:, 3:] = e_s_cropped + e_phi_cropped
# directors_2dcurved_cropped[:, 3:] /= np.linalg.norm(directors_2dcurved_cropped[:, 3:])
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
#     vec_dir=[directors_2dcurved_cropped[:, 3:], e_s, e_phi], vec_colors=["white", "orange", "purple"],
#     vec_names=["dir", "e_s", "e_phi"])
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[directors_2dcurved_cropped[:, :3], directors_2dcurved_cropped[:, :3]],
#     vec_dir=[e_s, e_phi], vec_colors=["orange", "lightblue"], vec_names=["e_s", "e_phi"])
# visuals.view_colored_mesh(gastr_mesh, vert_colors=visuals.color_scalar(
#     analysis.interpolate_on_mesh(values=S_2dcurv, mesh=gastr_mesh, value_idxs=idxs_sel), manual_vminmax=[0, 1],
#     cmap="Spectral"), mesh_shading="flat", mesh_blending="opaque", mesh_opacity=1.0)

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 40]
# patch_avg = ["nearest", 20]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
plot2d_view = (20, 0)
histfigsize = (4, 3)
renderfigsize = (6, 5)
veccoords = directors_2dcurved_cropped[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Calculate Curved Nematic Order ====
S_2dcurv_sphi_cropped, n_avg_2dcurv_sphi_cropped, q_sphi_cropped = analysis.avg_tan_nem_tens(t1_cov=e_s_cropped,
                                                                                             t2_cov=e_phi_cropped,
                                                                                             directors=directors_2dcurved_cropped,
                                                                                             neigh_idxs=neigh_idxs,
                                                                                             return_qij_bar=True)

# ==== Save Curved Nematic Order ====
# datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
# datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
#                        header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_cropped_avg_sphi = directors_2dcurved_cropped.copy()
directors_2dcurved_cropped_avg_sphi[:, 3:] = n_avg_2dcurv_sphi_cropped
# savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.png")
# savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.png")
savefig_render = ""
savefig_hist = ""

visuals.plot_dir_field(directors=directors_2dcurved_cropped_avg_sphi, veclength=vec_length, view_init=plot2d_view,
                       veccolor=S_2dcurv_sphi_cropped, cmap_label="order scalar $S$", title=title_render,
                       manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv_sphi_cropped, title=title_hist, savefig=savefig_hist, figsize=histfigsize, xlim=[0, 1])

s_bin_centers_cropped, Q_ss_cropped, Q_ss_mean_cropped, Q_phiphi_cropped, Q_phiphi_mean_cropped, Q_sphi_cropped, Q_sphi_mean_cropped = extra_gastruloids.decompose_q_sphi(
    q_sphi=q_sphi_cropped, s_coords=dir_s_cropped, num_bins=50)
visuals.plot_qsphi_profiles(dir_s=dir_s_cropped, s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                            Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                            Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped, Q_sphi_mean=Q_sphi_mean_cropped,
                            y_limits=[-0.5, 0.5])

visuals.plot_qsphi_profiles_separated_phi(dir_s=dir_s_cropped, dir_phi=dir_phi_cropped,
                                          s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                                          Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                                          Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped,
                                          Q_sphi_mean=Q_sphi_mean_cropped,
                                          y_limits=[-0.5, 0.5])


In [ ]:
visuals.plot_qsphi_profiles(dir_s=dir_s_cropped, s_bin_centers=s_bin_centers_cropped, Q_ss=Q_ss_cropped,
                            Q_ss_mean=Q_ss_mean_cropped, Q_phiphi=Q_phiphi_cropped,
                            Q_phiphi_mean=Q_phiphi_mean_cropped, Q_sphi=Q_sphi_cropped, Q_sphi_mean=Q_sphi_mean_cropped,
                            y_limits=[-0.5, 0.5])

In [ ]:
visuals.plot_s_sphi_profile(dir_s=dir_s_cropped, s_global=S_2dcurv_sphi_cropped, y_limits=[0, 1],
                            savefig=f"/Users/andreadi/Desktop/112h_200_Gas2_global-s.png")

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 1
visuals.view_colored_mesh_dir_field(mesh=gastr_mesh, directors=directors_2dcurved_cropped,
                                    vec_colors=visuals.color_scalar(S_2dcurv_sphi_cropped, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(analysis.normalise_range(proj_layer),
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)

## -- Plot decompositions --

In [ ]:
path_all = [
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/72h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/96h_300_Gas3.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/104h_300_Gas1.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/112h_300_Gas2.tif",
    "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/120h_300_Gas9.tif"]
projection_label = "proj_-36.0_to_-35.0_um"  #"proj_-21.0_to_-20.0_um"
time_points_all = []
all_packets = []
reload(extra_gastruloids)
for path in path_all:
    decomposition_packet = extra_gastruloids.proj_nem_on_sphi(img_path=path, layer_label=projection_label)
    all_packets.append(decomposition_packet)
    time_points_all.append(int(path.split("h_")[0].split("/")[-1]))
s_bin_centers_all = []
Q_ss_all = []
Q_phiphi_all = []
Q_sphi_all = []
Q_ss_mean_all = []
Q_phiphi_mean_all = []
Q_sphi_mean_all = []
dir_s_all = []
dir_phi_all = []
dir_rho_all = []
mesh_s_all = []
mesh_phi_all = []
mesh_rho_all = []
for i, packet in enumerate(all_packets):
    mesh_coords, dir_coords, q_decomposition = packet
    s_bin_centers, Q_ss, Q_ss_mean, Q_phiphi, Q_phiphi_mean, Q_sphi, Q_sphi_mean = q_decomposition
    dir_s, dir_rho, dir_phi = dir_coords
    mesh_s, mesh_rho, mesh_phi = mesh_coords
    s_bin_centers_all.append(s_bin_centers)
    Q_ss_all.append(Q_ss)
    Q_phiphi_all.append(Q_phiphi)
    Q_sphi_all.append(Q_sphi)
    Q_ss_mean_all.append(Q_ss_mean)
    Q_phiphi_mean_all.append(Q_phiphi_mean)
    Q_sphi_mean_all.append(Q_sphi_mean)
    dir_s_all.append(dir_s)
    dir_phi_all.append(dir_phi)
    dir_rho_all.append(dir_rho)
    mesh_s_all.append(mesh_s)
    mesh_phi_all.append(mesh_phi)
    mesh_rho_all.append(mesh_rho)

mesh_s_all_min = [np.min(s) for s in mesh_s_all]
mesh_s_all_max = [np.max(s) for s in mesh_s_all]
mesh_rho_all_min = [np.min(rho) for rho in mesh_rho_all]
mesh_rho_all_max = [np.max(rho) for rho in mesh_rho_all]
mesh_s_all_mean = [np.mean(s) for s in mesh_s_all]
mesh_rho_all_mean = [np.mean(rho) for rho in mesh_rho_all]

In [ ]:
plt.figure()
plt.plot(time_points_all, sphericity, "o-", c="g")
plt.title(r"Sphericity Over Time: $\Psi = \frac{\pi^{1/3}(6V)^{2/3}}{A}$")
plt.xlabel("Time (h)")
plt.yticks(np.arange(0, 1.1, 0.1))
plt.grid()
plt.ylim(0, 1)
plt.show()
plt.figure()
plt.plot(time_points_all, [s.max() / (2 * rho.max()) for s, rho in zip(mesh_s_all, mesh_rho_all)], "o-", c="g")
plt.title("A<->P body length / maximum diameter")
plt.xlabel("Time (h)")
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_max, "o-", c="r", label="max arc length")
plt.plot(time_points_all, mesh_rho_all_max, "o-", c="b", label="max diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.legend()
plt.show()
plt.figure()
plt.plot(time_points_all, mesh_s_all_mean, "o-", c="r", label="mean arc length")
plt.plot(time_points_all, mesh_rho_all_mean, "o-", c="b", label="mean diameter")
plt.xlabel("Time (h)")
plt.ylabel(r"Length ($\mu$m)")
plt.yscale("log")
plt.legend()
plt.show()

In [ ]:
img_unit = "um"
normalise_bodyaxis = False
cbarlabel = "Developmental time (h)"
ylabel = r"$\rho$" + f" ({img_unit})"
title = r"Thickness $\rho$ Profile" + f" ({img_unit})"
if normalise_bodyaxis:
    mesh_s_norm_all = []
    for i in range(len(mesh_s_all)):
        s_min, s_max = mesh_s_all[i].min(), mesh_s_all[i].max()
        s_norm = (mesh_s_all[i] - s_min) / (s_max - s_min)
        mesh_s_norm_all.append(s_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_norm_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_thickness_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_rho_profile_evolution(time_points_all=time_points_all, mesh_s_all=mesh_s_all,
                                       mesh_rho_all=mesh_rho_all, xlabel=xlabel, title=title, ylabel=ylabel,
                                       cbarlabel=cbarlabel, savefig=savefig)

In [ ]:
img_unit = "um"
normalise_bodyaxis = True
cbarlabel = "Developmental time (h)"
title = r"Nematic order $Q_{s\phi}$ Profile" + f" \n {projection_label}"
q_mean_all_dict = {
    "Q_ss": Q_ss_mean_all,
    "Q_phiphi": Q_phiphi_mean_all,
    "Q_sphi": Q_sphi_mean_all,
}
if normalise_bodyaxis:
    s_bin_centers_norm_all = []
    for i in range(len(dir_s_all)):
        s_min, s_max = dir_s_all[i].min(), dir_s_all[i].max()
        s_bin_norm = (s_bin_centers_all[i] - s_min) / (s_max - s_min)
        s_bin_centers_norm_all.append(s_bin_norm)
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile_normalised-body-axis.png"
    xlabel = "Normalised arc length $s/L$"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_norm_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])
else:
    savefig = f"/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/1_gastruloid/!2025_corrected-membrane/{projection_label}_q-s-phi_profile.png"
    xlabel = r"Arc length $s$" + f" ({img_unit})"
    visuals.plot_qsphi_profile_evolution(time_points_all=time_points_all, s_bin_centers_all=s_bin_centers_all,
                                         q_mean_all_dict=q_mean_all_dict,
                                         xlabel=xlabel, title=title, cbarlabel=cbarlabel, savefig=savefig,
                                         ylim=[-0.5, 0.5])

## BATCH EXTRACTION

In [ ]:
img_path_list_72h = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/200/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/200/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/200/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/300/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/300/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/72/300/Gas3.tif']

img_path_list_96h = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/200/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas6.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas7.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas8.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/96/300/Gas9.tif']

img_path_list_104h = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas6.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas7.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas8.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas9.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/200/Gas10.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/300/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/300/Gas6.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/104/300/Gas7.tif']

img_path_list_112h = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/200/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas3.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas6.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas7.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas8.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas9.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/112/300/Gas10.tif']

img_path_list_120h = [
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/200/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/200/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/200/Gas4.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/200/Gas5.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/300/Gas1.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/300/Gas2.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/300/Gas10.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/300/Gas11.tif',
    '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/120/300/Gas12.tif']
all_img_paths = img_path_list_72h + img_path_list_96h + img_path_list_104h + img_path_list_112h + img_path_list_120h

In [ ]:
from batch_analysis.automated_scripts import run_cylindrical_analysis, run_projection, run_tan_orient_extract

In [ ]:
for path in all_img_paths:
    timepoint = int(path.split("_CLEAN/")[1].split("/")[0])
    print(f"[<<<<<>>>>>] TIME = {timepoint} [!!!!!!!] ")
    run_cylindrical_analysis.main(img_path=path, overwrite=True, show_figures=True,
                                  voxel_size=2 if timepoint == 72 else 3,
                                  spline_smooth_factor=400,
                                  render=False, force_pca_use=True if timepoint == 72 else False)

In [ ]:
min_dist_all = 15 - 0.05
max_dist_all = 15 + 0.05
num_dist_all = 30
img_unit = "um"
layer_label_all = f"proj_{min_dist_all}_to_{max_dist_all}_{img_unit}"
proj_mode_all = "mean"
for path in all_img_paths:
    print(f"=========== {path} ===========")
    run_projection.main(img_path=path, flip_normals=True, dist_min=min_dist_all, dist_max=max_dist_all,
                        dist_num=num_dist_all,
                        proj_mode="mean", render=False, show_plots=False)

In [ ]:
patch_size_all = 30
for path in all_img_paths:
    print(f"=========== {path} ===========")
    run_tan_orient_extract.main(img_path=path, layer_label=layer_label_all, render=False, patch_mode="radius",
                                patch_size=patch_size_all, normal_validity_k=20, normal_validity_thresh=0.99,
                                compute_num=3000, grid_n_2dcurve_analysis=30, debug_2dcurve_analysis=False,
                                show_figures=False)

## BATCH NEMATIC ANALYSIS

In [ ]:
import pandas as pd
import numpy as np
import os

all_profiles_list = []

for path in all_img_paths:
    print(f"=========== {path} ===========")
    path_parts = path.split(os.sep)
    gas_id = path_parts[-1].replace(".tif", "")
    size_val = path_parts[-2]
    t_val = int(path_parts[-3])
    print(f"Time: {t_val}h | Size: {size_val} | ID: {gas_id}")
    decomposition_packet = extra_gastruloids.proj_nem_on_sphi(img_path=path, layer_label=layer_label_all,
                                                              low_cutoff_phi=-np.pi / 3, high_cutoff_phi=np.pi / 3,
                                                              profile_bins=20)
    mesh_coords, dir_coords, q_decomposition = decomposition_packet
    s_bin_centers, Q_ss, Q_ss_mean, Q_phiphi, Q_phiphi_mean, Q_sphi, Q_sphi_mean = q_decomposition
    for j in range(len(s_bin_centers)):
        all_profiles_list.append({
            "Time Point": t_val,
            "Size": size_val,
            "Gas ID": gas_id,
            "s": s_bin_centers[j],
            "Q_ss": Q_ss_mean[j],
            "Q_phiphi": Q_phiphi_mean[j],
            "Q_sphi": Q_sphi_mean[j],
            "depth": layer_label_all
        })
df_nematic = pd.DataFrame(all_profiles_list)
print("Profile DataFrame Ready!")

In [ ]:
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
nematic_csv_path = os.path.join(desktop_path, "df_nematic_profiles.csv")
df_nematic.to_csv(nematic_csv_path, index=False)
print(f"Success! Nematic profiles saved to: {nematic_csv_path}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Plotting Q_ss (Order along the long axis)
sns.lineplot(
    data=df_nematic,
    x="s", y="Q_ss",
    hue="Time Point",
    style="Size",
    palette="viridis"
)

plt.title(fr"$\text{{Nematic Order }} Q_{{ss}} \text{{ along Arc Length }} s$" + f"{layer_label_all}")
plt.xlabel(r"Arc Length $s$ ($\mu m$)")
plt.ylabel(r"Order Parameter $Q_{ss}$")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# Create a new column 's_norm' by dividing s by the max s of each unique Gas ID
df_nematic['s_norm'] = df_nematic.groupby('Gas ID')['s'].transform(lambda x: x / x.max())
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Plotting Q_ss against normalized s
sns.lineplot(
    data=df_nematic,
    x="s_norm", y="Q_ss",
    hue="Time Point",
    style="Size",
    palette="viridis",
    # err_style="band" is default, showing the confidence interval
)

plt.title(
    fr"$\text{{Nematic Order }} Q_{{ss}} \text{{ along Normalized Arc Length }} s/s_{{max}}$" + f"\n{layer_label_all}")
plt.xlabel(r"Normalized Arc Length ($s/s_{max}$)")
plt.ylabel(r"Order Parameter $Q_{ss}$")

# Set x-axis limits to exactly 0 and 1
plt.xlim(0, 1)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
sns.despine()  # Clean up the top/right spines
plt.tight_layout()
plt.show()

In [ ]:



def plot_nematic_evolution_proper(df, figsize=(10, 6), n_grid=300, sigma=10):
    # 1. Define conditions
    time_points = np.sort(df['Time Point'].unique())
    sizes = np.sort(df['Size'].unique())

    cmap = cm.coolwarm(np.linspace(0, 1, len(time_points)))
    time_to_color = {t: cmap[i] for i, t in enumerate(time_points)}
    style_map = {sz: style for sz, style in zip(sizes, ['-', '--', ':', '-.'])}

    fig, ax = plt.subplots(figsize=figsize)
    s_norm_grid = np.linspace(0, 1, n_grid)

    for sz in sizes:
        for t in time_points:
            # Mask for this specific cohort
            cohort = df[(df['Time Point'] == t) & (df['Size'] == sz)]
            if cohort.empty: continue

            group_qss = []

            # Iterate through unique gastruloids in this specific cohort
            for gid in cohort['Gas ID'].unique():
                sample = cohort[cohort['Gas ID'] == gid].sort_values('s')

                s_phys = sample['s'].values
                q_vals = sample['Q_ss'].values

                if len(s_phys) < 5: continue

                # Spatial normalization (0 to 1) for this individual
                s_norm = (s_phys - s_phys.min()) / (s_phys.max() - s_phys.min())

                # Map this individual to the master grid
                f = interp1d(s_norm, q_vals, bounds_error=False, fill_value="extrapolate")
                group_qss.append(f(s_norm_grid))

            if not group_qss: continue

            # Population statistics for this cohort
            mean_q = np.mean(group_qss, axis=0)
            sem_q = np.std(group_qss, axis=0) / np.sqrt(len(group_qss))  # SEM is better for averages

            # Gaussian smoothing for publication-quality lines
            mean_smooth = gaussian_filter1d(mean_q, sigma=sigma, mode='nearest')
            sem_smooth = gaussian_filter1d(sem_q, sigma=sigma, mode='nearest')

            color = time_to_color[t]
            ls = style_map[sz]

            # Plotting the population "state" at this time point
            ax.plot(s_norm_grid, mean_smooth, color=color, linestyle=ls, linewidth=2.5)
            ax.fill_between(s_norm_grid, mean_smooth - sem_smooth, mean_smooth + sem_smooth,
                            color=color, alpha=0.15, linewidth=0)

    # --- Polish ---
    ax.set_xlim(0, 1)
    ax.set_xlabel(r"Normalized Arc Length ($s/s_{\max}$)", fontsize=12)
    ax.set_ylabel(r"Order Parameter $Q_{ss}$", fontsize=12)
    ax.set_title(r"$\text{Nematic Order Evolution: Snapshot Population Averages}$", fontsize=14, pad=15)

    ax.grid(True, alpha=0.1)
    sns.despine()

    # Custom Legend for Sizes
    from matplotlib.lines import Line2D
    size_handles = [Line2D([0], [0], color='gray', linestyle=st, label=f"Size {sz}") for sz, st in style_map.items()]
    ax.legend(handles=size_handles, loc='upper left', bbox_to_anchor=(1.05, 1), title="Seeding Size", frameon=False)

    # Colorbar for Time
    sm = cm.ScalarMappable(cmap=ListedColormap(cmap),
                           norm=BoundaryNorm(np.arange(len(time_points) + 1) - 0.5, len(time_points)))
    sm.set_array([])
    cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax, ticks=np.arange(len(time_points)))
    cbar.set_label(r"Time point $t$ (h)")
    cbar.ax.set_yticklabels([str(int(tp)) for tp in time_points])

    plt.tight_layout()
    plt.show()


plot_nematic_evolution_proper(df_nematic, sigma=12)

In [ ]:



def plot_nematic_amplitude_smooth(df, figsize=(10, 6), n_grid=300, sigma=10):
    # 1. Calculate the Scalar Order Parameter S
    # S = 2 * sqrt(Qss^2 + Qsphi^2)
    # We use the factor of 2 to normalize S between [0, 1] for 2D nematics
    df['S_amp'] = 2 * np.sqrt(df['Q_ss'] ** 2 + df['Q_sphi'] ** 2)

    time_points = np.sort(df['Time Point'].unique())
    sizes = np.sort(df['Size'].unique())

    cmap = cm.coolwarm(np.linspace(0, 1, len(time_points)))
    time_to_color = {t: cmap[i] for i, t in enumerate(time_points)}
    style_map = {sz: style for sz, style in zip(sizes, ['-', '--', ':', '-.'])}

    fig, ax = plt.subplots(figsize=figsize)
    s_norm_grid = np.linspace(0, 1, n_grid)

    for sz in sizes:
        for t in time_points:
            cohort = df[(df['Time Point'] == t) & (df['Size'] == sz)]
            if cohort.empty: continue

            group_S = []
            for gid in cohort['Gas ID'].unique():
                sample = cohort[cohort['Gas ID'] == gid].sort_values('s')
                s_phys = sample['s'].values
                s_val = sample['S_amp'].values  # The new amplitude column

                if len(s_phys) < 5: continue

                s_norm = (s_phys - s_phys.min()) / (s_phys.max() - s_phys.min())
                f = interp1d(s_norm, s_val, bounds_error=False, fill_value="extrapolate")
                group_S.append(f(s_norm_grid))

            if not group_S: continue

            # Population Stats
            mean_S = np.mean(group_S, axis=0)
            sem_S = np.std(group_S, axis=0) / np.sqrt(len(group_S))

            # Smoothing
            mean_smooth = gaussian_filter1d(mean_S, sigma=sigma, mode='nearest')
            sem_smooth = gaussian_filter1d(sem_S, sigma=sigma, mode='nearest')

            color = time_to_color[t]
            ls = style_map[sz]

            ax.plot(s_norm_grid, mean_smooth, color=color, linestyle=ls, linewidth=2.5)
            ax.fill_between(s_norm_grid, mean_smooth - sem_smooth, mean_smooth + sem_smooth,
                            color=color, alpha=0.15, linewidth=0)

    # --- Formatting ---
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)  # Amplitude is strictly [0, 1]
    ax.set_xlabel(r"Normalized Arc Length ($s/s_{\max}$)", fontsize=12)
    ax.set_ylabel(r"Nematic Amplitude $S$", fontsize=12)
    ax.set_title(r"$\text{Global Nematic Order Parameter } S \text{ Evolution}$", fontsize=14, pad=15)

    ax.grid(True, alpha=0.1)
    sns.despine()

    # Legend & Colorbar
    from matplotlib.lines import Line2D
    size_handles = [Line2D([0], [0], color='gray', linestyle=st, label=f"Size {sz}") for sz, st in style_map.items()]
    ax.legend(handles=size_handles, loc='upper left', bbox_to_anchor=(1.05, 1), title="Seeding Size", frameon=False)

    sm = cm.ScalarMappable(cmap=ListedColormap(cmap),
                           norm=BoundaryNorm(np.arange(len(time_points) + 1) - 0.5, len(time_points)))
    sm.set_array([])
    cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.1)
    cbar = plt.colorbar(sm, cax=cax, ticks=np.arange(len(time_points)))
    cbar.set_label(r"Time point $t$ (h)")
    cbar.ax.set_yticklabels([str(int(tp)) for tp in time_points])

    plt.tight_layout()
    plt.show()


# --- RUN ---
plot_nematic_amplitude_smooth(df_nematic, sigma=12)

In [ ]:



def plot_nematic_analysis(df, metric='Q_global', sigma=12, n_grid=300, figsize=(10, 6)):
    # 1. Configuration with corrected LaTeX (No nested dollar signs)
    metric_configs = {
        'Q_ss': {'col': 'Q_ss', 'ylab': r'$Q_{ss}$', 'ylim': (-0.6, 0.6), 'flip': False},
        'Q_sphi': {'col': 'Q_sphi', 'ylab': r'$Q_{s\phi}$', 'ylim': (-0.6, 0.6), 'flip': False},
        'Q_phiphi': {'col': 'Q_ss', 'ylab': r'$Q_{\phi\phi}$', 'ylim': (-0.6, 0.6), 'flip': True},
        'Q_global': {'col': 'S_amp', 'ylab': r'Order Scalar $S$', 'ylim': (0, 1), 'flip': False}
    }

    if metric not in metric_configs:
        raise ValueError(f"Metric must be one of {list(metric_configs.keys())}")

    config = metric_configs[metric]

    # 2. Safety copy and calculation
    df = df.copy()
    if 'S_amp' not in df.columns:
        df['S_amp'] = 2 * np.sqrt(df['Q_ss'] ** 2 + df['Q_sphi'] ** 2)

    time_points = np.sort(df['Time Point'].unique())
    cmap = cm.coolwarm(np.linspace(0, 1, len(time_points)))
    time_to_color = {t: cmap[i] for i, t in enumerate(time_points)}

    fig, ax = plt.subplots(figsize=figsize)
    s_norm_grid = np.linspace(0, 1, n_grid)

    for t in time_points:
        cohort = df[df['Time Point'] == t]
        if cohort.empty: continue

        group_data = []
        for gid in cohort['Gas ID'].unique():
            sample = cohort[cohort['Gas ID'] == gid].sort_values('s')
            s_phys = sample['s'].values
            y_vals = sample[config['col']].values

            if config['flip']: y_vals = -y_vals
            if len(s_phys) < 5: continue

            s_norm = (s_phys - s_phys.min()) / (s_phys.max() - s_phys.min())
            f = interp1d(s_norm, y_vals, bounds_error=False, fill_value="extrapolate")
            group_data.append(f(s_norm_grid))

        if not group_data: continue

        mean_y = np.mean(group_data, axis=0)
        sem_y = np.std(group_data, axis=0) / np.sqrt(len(group_data))

        mean_smooth = gaussian_filter1d(mean_y, sigma=sigma, mode='nearest')
        sem_smooth = gaussian_filter1d(sem_y, sigma=sigma, mode='nearest')

        color = time_to_color[t]
        ax.plot(s_norm_grid, mean_smooth, color=color, linewidth=3, label=f"{t}h")
        ax.fill_between(s_norm_grid, mean_smooth - sem_smooth, mean_smooth + sem_smooth,
                        color=color, alpha=0.15, linewidth=0)

    # --- Formatting ---
    ax.set_xlim(0, 1)
    ax.set_ylim(config['ylim'])
    ax.set_xlabel(r"Normalized Arc Length ($s/s_{\max}$)", fontsize=12)
    ax.set_ylabel(config['ylab'], fontsize=12)

    # Use standard text for the title prefix to avoid TeX nesting errors
    ax.set_title(f"Evolution of {config['ylab']}", fontsize=14, pad=15)

    ax.grid(True, alpha=0.1)
    sns.despine()

    # Colorbar logic (Only if multiple time points exist)
    if len(time_points) > 1:
        sm = cm.ScalarMappable(cmap=ListedColormap(cmap),
                               norm=BoundaryNorm(np.arange(len(time_points) + 1) - 0.5, len(time_points)))
        sm.set_array([])
        cax = make_axes_locatable(ax).append_axes("right", size="3%", pad=0.1)
        cbar = plt.colorbar(sm, cax=cax, ticks=np.arange(len(time_points)))
        cbar.set_label("Time point (h)")
        cbar.ax.set_yticklabels([str(int(tp)) for tp in time_points])

    plt.tight_layout()
    plt.show()

In [ ]:

# plot_nematic_amplitude_pooled(df_nematic, sigma=15)
plot_nematic_analysis(
    df_nematic[(df_nematic["Size"] == "200") & (df_nematic["Time Point"] == 104) & (df_nematic["Gas ID"] == "Gas10")],
    metric="Q_global",
    sigma=15)

In [ ]:
df_nematic["depth"].unique()[0]

In [ ]:
plot_nematic_analysis(
    df_nematic[(df_nematic["Size"] == "200")],
    metric="Q_global", sigma=15)

## BATCH MORPHO ANALYSIS

In [ ]:
gastruloid_analysis_fig_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/'

In [ ]:
import os
import pandas as pd
import numpy as np

# --- 1. SETUP & PATHS ---
root_meshes_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/ALL_SAMPLING-MESHES'
gastruloid_analysis_fig_path = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/URI_25022026_PR_NEMO/EXP4_filter_membrane/'

root_meshes = [
    entry.path for entry in os.scandir(root_meshes_path)
    if entry.is_file() and entry.name.endswith(".ply")
]

# Add any specific (Time, Size, GasID) tuples here to exclude them
exclusion_list = []
exclusions = {(str(x[0]), str(x[1]), x[2]) for x in exclusion_list}

# Containers for Morphometrics and Profiles
data_list = []
mesh_s_all = []
mesh_rho_all = []
time_points_all = []
size_labels_all = []

print(f"Processing {len(root_meshes)} gastruloids...")

# --- 2. DATA EXTRACTION LOOP ---
for fname in root_meshes:
    base = os.path.basename(fname).replace(".ply", "")
    parts = base.split("_")
    # parts[0]=Time, parts[1]=Size, parts[2]=GasID
    current_identity = (parts[0], parts[1], parts[2])

    if current_identity in exclusions:
        continue

    prefix = base.replace("sampling_mesh", "")
    folder = os.path.dirname(fname)

    try:
        # Load Mesh for global metrics
        mesh = datahandler.load_mesh(filepath=fname, recalc_normals=False, clean=False)

        # Load Profile Arrays (s, rho, phi)
        profile_path = os.path.join(os.path.dirname(folder), "ALL_MORPHO_PROFILES")
        mesh_s_rho_phi = datahandler.load_array(name=prefix + "mesh_s-rho-phi", folderpath=profile_path)

        s = mesh_s_rho_phi[:, 0]
        rho = mesh_s_rho_phi[:, 1]

        # Calculate Mid Body Thickness
        mid_s_target = s.max() / 2
        mid_idx = (np.abs(s - mid_s_target)).argmin()
        mid_thickness = rho[mid_idx]

        # Store for df_morpho
        record = {
            "File": base,
            "Time Point": int(parts[0]),
            "Size": parts[1],
            "Area": mesh.area,
            "Volume": mesh.volume,
            "Body Length": s.max(),
            "Average Body Thickness": np.mean(rho),
            "Max Body Thickness": np.max(rho),
            "Mid Body Thickness": mid_thickness
        }
        data_list.append(record)

        # Store for Profile Plotter
        mesh_s_all.append(s)
        mesh_rho_all.append(rho)
        time_points_all.append(int(parts[0]))
        size_labels_all.append(parts[1])

    except Exception as e:
        print(f"Error processing {base}: {e}")

df_morpho = pd.DataFrame(data_list)
print(f"DataFrame Ready! {len(df_morpho)} samples retained.")


In [ ]:
# --- 3. SAVE TO DESKTOP ---
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")

# A. Save the Morphometrics DataFrame as CSV
csv_out = os.path.join(desktop_path, "gastruloid_morphometrics.csv")
df_morpho.to_csv(csv_out, index=False)

# B. Save the Profile Lists as a compressed NPZ
# We use dtype=object because each 's' and 'rho' array has a different length
npz_out = os.path.join(desktop_path, "gastruloid_profiles.npz")
np.savez_compressed(
    npz_out,
    mesh_s_all=np.array(mesh_s_all, dtype=object),
    mesh_rho_all=np.array(mesh_rho_all, dtype=object),
    time_points_all=np.array(time_points_all),
    size_labels_all=np.array(size_labels_all)
)

print(f"--- Files Saved to Desktop ---")
print(f"1. CSV: {csv_out}")
print(f"2. NPZ: {npz_out}")

In [ ]:



def plot_dual_rho_evolution(time_points, s_list, rho_list, figsize=(10, 10), title_suffix=""):
    t_array = np.array(time_points)
    time_unique = np.sort(np.unique(t_array))
    cmap = cm.coolwarm(np.linspace(0, 1, len(time_unique)))

    # Create 1 row, 2 columns
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
    plt.subplots_adjust(right=0.9, top=0.88, wspace=0.25)

    for idx, tp in enumerate(time_unique):
        indices = np.where(t_array == tp)[0]

        # Lists to store points for bulk scattering
        all_s_phys = []
        all_s_norm = []
        all_rho = []

        for i in indices:
            s_orig = np.array(s_list[i])
            rho_orig = np.array(rho_list[i])

            if s_orig.max() == s_orig.min(): continue

            # Non-normalized
            all_s_phys.append(s_orig)
            # Normalized
            all_s_norm.append((s_orig - s_orig.min()) / (s_orig.max() - s_orig.min()))
            # Radius
            all_rho.append(rho_orig)

        if all_s_phys:
            # Plot Physical Scale
            ax1.scatter(np.concatenate(all_s_phys), np.concatenate(all_rho),
                        s=1, alpha=0.002, color=cmap[idx], rasterized=True)
            # Plot Normalized Scale
            ax2.scatter(np.concatenate(all_s_norm), np.concatenate(all_rho),
                        s=1, alpha=0.002, color=cmap[idx], rasterized=True)

    # --- Formatting Ax 1 (Physical) ---
    ax1.set_xlabel(r"Arc Length $s$ ($\mu m$)")
    ax1.set_ylabel(r"Radius $\rho$ ($\mu m$)", fontsize=12)
    ax1.grid(True, alpha=0.2)

    # --- Formatting Ax 2 (Normalized) ---
    ax2.set_xlabel(r"Normalized Arc Length ($s/s_{max}$)")
    ax2.set_xlim(0, 1)
    ax2.grid(True, alpha=0.2)

    fig.suptitle(fr"$\text{{Thickness Profile Evolution {title_suffix}}}$", fontsize=16, y=0.98)
    sns.despine()

    # --- Unified Colorbar ---
    cmap_listed = ListedColormap(cmap)
    bounds = np.arange(len(time_unique) + 1) - 0.5
    sm = cm.ScalarMappable(cmap=cmap_listed, norm=BoundaryNorm(bounds, cmap_listed.N))
    sm.set_array([])

    # Place colorbar to the far right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(time_unique)))
    cbar.set_label(r"Time (h)", fontsize=12)
    cbar.ax.set_yticklabels([str(int(tp)) for tp in time_unique])
    plt.savefig(os.path.join(gastruloid_analysis_fig_path, "rho-profile-evolution.png"), bbox_inches="tight", dpi=300)
    plt.show()


# --- Execution ---
# One call generates the comparison for the whole dataset
plot_dual_rho_evolution(
    time_points=time_points_all,
    s_list=mesh_s_all,
    rho_list=mesh_rho_all
)

In [ ]:



def plot_fitted_spaghetti_ultra_smooth(time_points, s_list, rho_list, size_labels,
                                       figsize=(10, 10), n_grid=200, sigma=10):
    t_array = np.array(time_points)
    sz_array = np.array(size_labels)
    time_unique = np.sort(np.unique(t_array))
    size_unique = np.sort(np.unique(sz_array))

    cmap = cm.coolwarm(np.linspace(0, 1, len(time_unique)))
    time_to_color = {t: cmap[i] for i, t in enumerate(time_unique)}
    style_map = {size: style for size, style in zip(size_unique, ['-', '--', ':', '-.'])}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
    plt.subplots_adjust(right=0.85, top=0.92, hspace=0.3)

    s_norm_grid = np.linspace(0, 1, n_grid)

    print(f"Generating Gaussian-smoothed traces (sigma={sigma}) for {len(s_list)} gastruloids...")

    for i in range(len(s_list)):
        s_raw = np.array(s_list[i])
        rho_raw = np.array(rho_list[i])

        if len(s_raw) < 10 or s_raw.max() == s_raw.min():
            continue

        # 1. Sort
        sort_idx = np.argsort(s_raw)
        s_sorted = s_raw[sort_idx]
        rho_sorted = rho_raw[sort_idx]

        # 2. Resample to a fixed high-res internal grid BEFORE smoothing
        internal_grid = np.linspace(s_sorted.min(), s_sorted.max(), 500)
        rho_interp = np.interp(internal_grid, s_sorted, rho_sorted)

        # 3. Gaussian Smoothing
        rho_smoothed = gaussian_filter1d(rho_interp, sigma=sigma, mode='nearest')

        # 4. Final Interpolation for plotting grid
        f_final = interp1d(internal_grid, rho_smoothed, kind='cubic', fill_value="extrapolate")
        s_phys_grid = np.linspace(s_sorted.min(), s_sorted.max(), n_grid)
        rho_final = f_final(s_phys_grid)

        # Plot
        color = time_to_color[t_array[i]]
        ls = style_map[sz_array[i]]

        ax1.plot(s_phys_grid, rho_final, color=color, linestyle=ls, linewidth=1.5, alpha=0.4)
        ax2.plot(s_norm_grid, rho_final, color=color, linestyle=ls, linewidth=1.5, alpha=0.4)

    # --- Formatting with LaTeX labels ---
    max_s = max([s.max() for s in s_list])
    max_rho = max([r.max() for r in rho_list])

    # Subplot 1: Physical Scale
    ax1.set_xlim(0, max_s)
    ax1.set_ylim(0, max_rho * 1.1)
    ax1.set_xlabel(r"Arc Length $s$ ($\mu$m)", fontsize=12)
    ax1.set_ylabel(r"Radius $\rho$ ($\mu$m)", fontsize=12)

    # Subplot 2: Normalized Scale
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, max_rho * 1.1)
    ax2.set_xlabel(r"Normalized Arc Length ($s/s_{\max}$)", fontsize=12)
    ax2.set_ylabel(r"Radius $\rho$ ($\mu$m)", fontsize=12)

    for ax in [ax1, ax2]:
        ax.grid(True, alpha=0.15)
        sns.despine(ax=ax)

    # Legend & Colorbar
    from matplotlib.lines import Line2D
    size_handles = [Line2D([0], [0], color='gray', linestyle=st, label=f"Size {sz}") for sz, st in style_map.items()]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.9), title="Seeding Size", frameon=False)

    sm = cm.ScalarMappable(cmap=ListedColormap(cmap),
                           norm=BoundaryNorm(np.arange(len(time_unique) + 1) - 0.5, len(time_unique)))
    sm.set_array([])
    cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.6])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(time_unique)))
    cbar.set_label(r"Time point $t$ (h)", fontsize=12)
    cbar.ax.set_yticklabels([str(int(tp)) for tp in time_unique])
    plt.savefig(os.path.join(gastruloid_analysis_fig_path, "rho-profile-evolution_smooth.png"), bbox_inches="tight",
                dpi=300)

    plt.show()


# --- RUN ---
plot_fitted_spaghetti_ultra_smooth(time_points_all, mesh_s_all, mesh_rho_all, size_labels_all, sigma=10)

In [ ]:
import numpy as np
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
import os


def plot_averaged_profiles_smooth(time_points, s_list, rho_list, size_labels,
                                  figsize=(10, 10), n_grid=300, sigma=10):
    t_array = np.array(time_points)
    sz_array = np.array(size_labels)
    time_unique = np.sort(np.unique(t_array))
    size_unique = np.sort(np.unique(sz_array))

    cmap = cm.coolwarm(np.linspace(0, 1, len(time_unique)))
    time_to_color = {t: cmap[i] for i, t in enumerate(time_unique)}
    style_map = {size: style for size, style in zip(size_unique, ['-', '--', ':', '-.'])}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
    plt.subplots_adjust(right=0.85, top=0.92, hspace=0.3)

    # 1. Normalized grid for averaging shape
    s_norm_grid = np.linspace(0, 1, n_grid)

    print(f"Calculating averaged profiles for {len(size_unique)} sizes and {len(time_unique)} time points...")

    for sz in size_unique:
        for t in time_unique:
            # Find all gastruloids for this specific size and time
            indices = np.where((t_array == t) & (sz_array == sz))[0]
            if len(indices) == 0:
                continue

            group_rhos_norm = []
            group_rhos_phys = []
            group_max_s = []

            for i in indices:
                s_raw = np.array(s_list[i])
                rho_raw = np.array(rho_list[i])

                if len(s_raw) < 10 or s_raw.max() == s_raw.min():
                    continue

                # Sort and unique for interpolation safety
                sort_idx = np.argsort(s_raw)
                s_sorted, unique_idx = np.unique(s_raw[sort_idx], return_index=True)
                rho_sorted = rho_raw[sort_idx][unique_idx]

                # Max length for physical averaging
                L = s_sorted.max()
                group_max_s.append(L)

                # Interpolate to common normalized grid
                f_norm = interp1d((s_sorted - s_sorted.min()) / L, rho_sorted,
                                  bounds_error=False, fill_value="extrapolate")
                group_rhos_norm.append(f_norm(s_norm_grid))

            if not group_rhos_norm:
                continue

            # 2. Calculate Means
            mean_rho_norm = np.mean(group_rhos_norm, axis=0)
            avg_L = np.mean(group_max_s)
            s_phys_grid = np.linspace(0, avg_L, n_grid)

            # 3. Apply Gaussian Smoothing to the Mean
            # This makes the "average" look very professional
            mean_rho_smooth = gaussian_filter1d(mean_rho_norm, sigma=sigma, mode='nearest')

            color = time_to_color[t]
            ls = style_map[sz]

            # 4. Plot Mean Lines
            ax1.plot(s_phys_grid, mean_rho_smooth, color=color, linestyle=ls, linewidth=2.5)
            ax2.plot(s_norm_grid, mean_rho_smooth, color=color, linestyle=ls, linewidth=2.5)

    # --- Formatting with LaTeX labels ---
    max_s = max([s.max() for s in s_list])
    max_rho = max([r.max() for r in rho_list])

    ax1.set_xlim(0, max_s)
    ax1.set_ylim(0, max_rho * 1.1)
    ax1.set_xlabel(r"Arc Length $s$ ($\mu$m)", fontsize=12)
    ax1.set_ylabel(r"Radius $\rho$ ($\mu$m)", fontsize=12)

    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, max_rho * 1.1)
    ax2.set_xlabel(r"Normalized Arc Length ($s/s_{\max}$)", fontsize=12)
    ax2.set_ylabel(r"Radius $\rho$ ($\mu$m)", fontsize=12)

    for ax in [ax1, ax2]:
        ax.grid(True, alpha=0.15)
        sns.despine(ax=ax)

    # Legend & Colorbar
    from matplotlib.lines import Line2D
    size_handles = [Line2D([0], [0], color='gray', linestyle=st, label=f"Size {sz}") for sz, st in style_map.items()]
    fig.legend(handles=size_handles, loc='upper right', bbox_to_anchor=(0.98, 0.9), title="Seeding Size", frameon=False)

    sm = cm.ScalarMappable(cmap=ListedColormap(cmap),
                           norm=BoundaryNorm(np.arange(len(time_unique) + 1) - 0.5, len(time_unique)))
    sm.set_array([])
    cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.6])
    cbar = fig.colorbar(sm, cax=cbar_ax, ticks=np.arange(len(time_unique)))
    cbar.set_label(r"Time point $t$ (h)", fontsize=12)
    cbar.ax.set_yticklabels([str(int(tp)) for tp in time_unique])
    plt.savefig(os.path.join(gastruloid_analysis_fig_path, "rho-profile-evolution_smooth_averaged-gastruloids.png"),
                bbox_inches="tight", dpi=300)

    plt.show()


# --- RUN ---
plot_averaged_profiles_smooth(time_points_all, mesh_s_all, mesh_rho_all, size_labels_all, sigma=8)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

metrics = ["Area", "Volume", "Body Length", "Max Body Thickness", "Average Body Thickness", "Mid Body Thickness"]
ylabels = [r"Area ($\mu m^2$)", r"Volume ($\mu m^3$)", r"Body Length ($\mu m$)",
           r"Max Body Thickness ($\mu m$)", r"Avg. Body Thickness ($\mu m$)", r"Mid Body Thickness ($\mu m$)"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes_flat = axes.ravel()

unique_sizes = sorted(df_morpho["Size"].unique())
colors = sns.color_palette("Set1", n_colors=len(unique_sizes))

for i, metric in enumerate(metrics):
    ax = axes_flat[i]
    for size, color in zip(unique_sizes, colors):
        subset = df_morpho[df_morpho["Size"] == size]
        sns.regplot(
            data=subset, x="Time Point", y=metric,
            ax=ax, color=color,
            scatter_kws={"s": 50, "edgecolor": "w", "alpha": 0.7, "linewidths": 0.5},
            line_kws={"linewidth": 2},
            label=f"Size {size}"
        )
    ax.set_title(fr"$\text{{Gastruloid {metric}}}$", pad=12)
    ax.set_ylabel(ylabels[i])

    # Add x-labels to all plots in the bottom row (indices 3, 4, 5)
    if i >= 3:
        ax.set_xlabel(r"Time point ($t$ [h])")

# Extract handles for the legend from the first plot
handles, labels = axes_flat[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))

# Place legend outside to the right so it doesn't block any of the 6 plots
fig.legend(by_label.values(), by_label.keys(),
           title=r"$\text{Seeding Size}$",
           loc='center right',
           bbox_to_anchor=(1.0, 0.5),
           frameon=False)

plt.tight_layout()
# Adjust to make room for the legend
plt.subplots_adjust(right=0.92, bottom=0.12)
plt.savefig(os.path.join(gastruloid_analysis_fig_path, "morphometrics.png"), bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

unique_sizes = sorted(df_morpho["Size"].unique())
colors = sns.color_palette("Set1", n_colors=len(unique_sizes))
fig, ax = plt.subplots(figsize=(6, 5))
sns.countplot(
    data=df_morpho,
    x="Time Point",
    hue="Size",
    palette="Set1",
    hue_order=unique_sizes,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.85,
    ax=ax
)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{int(height)}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center',
                    xytext=(0, 8),
                    textcoords='offset points',
                    fontsize=10,
                    weight='bold')
ax.set_title(r"$\text{Experimental Sample Size Overview}$", pad=20, fontsize=14)
ax.set_ylabel(r"$\text{Number of Gastruloids } (N)$", fontsize=12)
ax.set_xlabel(r"$\text{Time point } (t \text{ [h]})$", fontsize=12)
ax.legend(title=r"$\text{Seeding Size}$", frameon=False, loc='upper left')
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(gastruloid_analysis_fig_path, "sample_size_statistics.png"), bbox_inches='tight', dpi=300)
plt.show()